# Full Method 1 Notebook

This notebook is a self-contained, research-oriented implementation of **Method 1 (KG-GT)** from `method1.tex`.

It keeps the current project choice of **MSE-only practical training** by default:

- The architecture includes the Transformer and GAT path.
- `SoftDTW` is still inlined below for completeness.
- The default config uses `loss_lambda = 1.0`, so training remains resource-aware.

## Method 1 Audit

`docs/report.tex` includes Method 1 via `data/review/EEGtoEMG/Final/report/method1.tex`.

Implementation status against the report:

- EEG preprocessing: implemented
- EMG preprocessing: implemented
- Kinematic state vector: implemented
- CCA alignment: implemented
- Transformer temporal encoder: implemented
- Kinematic-guided GAT path: implemented
- Hybrid MSE + SoftDTW loss: implemented in code, but current config intentionally runs MSE-only
- Training/evaluation pipeline: implemented

The main missing deliverable was a **single self-contained notebook** for study, inspection, and paper-ready visualization. This notebook fills that gap.

In [ ]:
# @title EnvironmentAndImports
from __future__ import annotations

import copy
import io
import json
import logging
import math
import os
import random
import re
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import seaborn as sns
import torch
import torch.nn as nn
import yaml
from scipy.signal import butter, decimate, filtfilt, iirnotch, savgol_filter, sosfiltfilt
from sklearn.cross_decomposition import CCA
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.auto import tqdm
    _TQDM_AVAILABLE = True
except ImportError:
    _TQDM_AVAILABLE = False

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CONFIG_PATH = ROOT / "configs" / "default.yaml"
METHOD1_TEX_PATH = ROOT / "data" / "review" / "EEGtoEMG" / "Final" / "report" / "method1.tex"
logger = logging.getLogger("method1_notebook")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

In [ ]:
# @title LoadProjectConfig
with open(CONFIG_PATH, encoding="utf-8") as fh:
    cfg = yaml.safe_load(fh)

print("Config path:", CONFIG_PATH)
print("Model type:", cfg["model"]["type"])
print("Current loss_lambda:", cfg["training"]["loss_lambda"])
print("Participants:", cfg["data"]["participants"])
print("Window size / stride:", cfg["data"]["window_size"], cfg["data"]["stride"])

method1_text = METHOD1_TEX_PATH.read_text(encoding="utf-8")
print("Method 1 source loaded from:", METHOD1_TEX_PATH)
print("Contains GAT section:", "Kinematic-Guided Graph Attention Network" in method1_text)

In [ ]:
"""
Device selection and reproducible seeding for GPU training.

get_device() picks CUDA when available and falls back to CPU, so the same
code path runs on the GTX 1660 Ti and on a CPU-only machine. set_seed()
seeds Python, NumPy, and torch (CPU + CUDA) for reproducible runs.
"""

from __future__ import annotations

import logging
import os
import random

import numpy as np
import torch

logger = logging.getLogger(__name__)


def get_device(prefer_cuda: bool = True) -> torch.device:
    """Return the best available compute device.

    Args:
        prefer_cuda: When True (default), use CUDA if available.

    Returns:
        torch.device('cuda') if available and preferred, else torch.device('cpu').
    """
    if prefer_cuda and torch.cuda.is_available():
        device = torch.device("cuda")
        name = torch.cuda.get_device_name(0)
        total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        logger.info("Using CUDA device: %s (%.1f GB)", name, total_gb)
        return device

    if prefer_cuda:
        logger.warning(
            "CUDA not available (torch=%s). Falling back to CPU. "
            "If you expected GPU, reinstall the CUDA build: "
            "pip install torch==2.8.0 --index-url "
            "https://download.pytorch.org/whl/cu126",
            torch.__version__,
        )
    return torch.device("cpu")


def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    """Seed all RNGs for reproducibility.

    Args:
        seed: Random seed.
        deterministic: When True, force deterministic cuDNN kernels. This
            improves reproducibility at some throughput cost; set False to let
            cuDNN autotune for speed.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic

In [ ]:
"""
WAY-EEG-GAL .mat file loader.

Handles two file types:
  HS_PX_SY.mat  — continuous whole-series data (struct: hs)
  WS_PX_SY.mat  — windowed per-trial data      (struct: ws)

scipy.io.loadmat is called with squeeze_me=True, struct_as_record=False
so nested MATLAB structs are accessible as Python attributes.
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Union

import numpy as np
import scipy.io as sio

# ---------------------------------------------------------------------------
# Split definitions (series numbers per participant)
# ---------------------------------------------------------------------------

TRAIN_SERIES: list = list(range(1, 8))   # S1–S7
VAL_SERIES:   list = [8]                 # S8
TEST_SERIES:  list = [9]                 # S9
STAB_SERIES:  list = ["ST"]             # ST (stability / perturbation)

_SPLIT_MAP = {
    "train":     TRAIN_SERIES,
    "val":       VAL_SERIES,
    "test":      TEST_SERIES,
    "stability": STAB_SERIES,
    "all":       TRAIN_SERIES + VAL_SERIES + TEST_SERIES + STAB_SERIES,
}


def get_split_series(split: str) -> list:
    """Return series identifiers for the requested split.

    Args:
        split: one of 'train', 'val', 'test', 'stability', 'all'

    Returns:
        List of series identifiers (int or 'ST').
    """
    if split not in _SPLIT_MAP:
        raise ValueError(f"Unknown split '{split}'. Choose from {list(_SPLIT_MAP)}")
    return _SPLIT_MAP[split]


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _mat_load(path: Union[str, Path]) -> dict:
    """Load a .mat file with squeeze_me and struct_as_record=False."""
    return sio.loadmat(
        str(path),
        squeeze_me=True,
        struct_as_record=False,
        mat_dtype=False,
    )


def _to_f32(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr, dtype=np.float32)


def _names_to_list(names_field) -> list[str]:
    """Convert MATLAB cell/string array of channel names to Python list."""
    if isinstance(names_field, np.ndarray):
        flat = names_field.flatten()
        return [str(n).strip() for n in flat]
    if isinstance(names_field, str):
        return [names_field.strip()]
    return list(names_field)


def _series_id_from_filename(fname: str) -> Union[int, str]:
    """Extract series ID: 'HS_P1_S3.mat' -> 3, 'HS_P1_ST.mat' -> 'ST'."""
    name = Path(fname).stem.upper()
    match = re.search(r"_S(\d+|ST)$", name)
    if match is None:
        raise ValueError(f"Cannot parse series ID from filename: {fname}")
    raw = match.group(1)
    return int(raw) if raw.isdigit() else raw


# ---------------------------------------------------------------------------
# HS loader  (continuous whole-series)
# ---------------------------------------------------------------------------

def load_hs(path: Union[str, Path]) -> dict:
    """Load one HS_PX_SY.mat file.

    Returns a dict with:
        eeg        : ndarray (T_eeg, 32)   float32, uV
        emg        : ndarray (T_emg, 5)    float32, raw units
        kin        : ndarray (T_kin, 36)   float32, mm / N / N*mm
        fs_eeg     : int    sampling rate EEG  (500 Hz)
        fs_emg     : int    sampling rate EMG  (4000 Hz)
        fs_kin     : int    sampling rate kin  (500 Hz)
        eeg_names  : list[str]  32 channel labels
        emg_names  : list[str]  5 muscle labels
        kin_names  : list[str]  36 kinematic channel labels
        participant: int
        series     : int | str  (numeric or 'ST')
        path       : str
    """
    path = Path(path)
    mat = _mat_load(path)
    hs = mat["hs"]

    eeg_sig = _to_f32(hs.eeg.sig)
    emg_sig = _to_f32(hs.emg.sig)
    kin_sig = _to_f32(hs.kin.sig)

    # Ensure 2-D even if single channel
    if eeg_sig.ndim == 1:
        eeg_sig = eeg_sig[:, None]
    if emg_sig.ndim == 1:
        emg_sig = emg_sig[:, None]
    if kin_sig.ndim == 1:
        kin_sig = kin_sig[:, None]

    fs_eeg = int(np.asarray(hs.eeg.samplingrate).flat[0])
    fs_emg = int(np.asarray(hs.emg.samplingrate).flat[0])
    fs_kin = int(np.asarray(hs.kin.samplingrate).flat[0])

    eeg_names = _names_to_list(hs.eeg.names)
    emg_names = _names_to_list(hs.emg.names)
    kin_names = _names_to_list(hs.kin.names)

    participant = int(np.asarray(hs.participant).flat[0])
    series = _series_id_from_filename(path.name)

    return {
        "eeg":         eeg_sig,
        "emg":         emg_sig,
        "kin":         kin_sig,
        "fs_eeg":      fs_eeg,
        "fs_emg":      fs_emg,
        "fs_kin":      fs_kin,
        "eeg_names":   eeg_names,
        "emg_names":   emg_names,
        "kin_names":   kin_names,
        "participant": participant,
        "series":      series,
        "path":        str(path),
    }


# ---------------------------------------------------------------------------
# WS loader  (windowed per-trial)
# ---------------------------------------------------------------------------

def _scalar(x) -> float:
    return float(np.asarray(x).flat[0])


def load_ws(path: Union[str, Path]) -> list[dict]:
    """Load one WS_PX_SY.mat file.

    Returns a list of trial dicts (one per lift), each with:
        eeg             : ndarray (T_eeg, 32)  float32
        emg             : ndarray (T_emg, 5)   float32
        kin             : ndarray (T_kin, 45)  float32  (36 raw + 9 derived)
        eeg_t           : ndarray (T_eeg,)     time vector (s)
        emg_t           : ndarray (T_emg,)     time vector (s)
        weight          : int    1=165g, 2=330g, 4=660g
        surf            : int    1=sandpaper, 2=suede, 3=silk
        weight_id       : str
        surf_id         : str
        led_on          : float  (s, relative to trial window)
        led_off         : float
        trial_start_time: float  (s, absolute within series)
        trial_idx       : int    0-based index within series
        participant     : int
        series          : int | str
        path            : str
    """
    path = Path(path)
    mat = _mat_load(path)
    ws = mat["ws"]

    wins = ws.win
    if not hasattr(wins, "__len__"):
        wins = [wins]

    participant = int(np.asarray(ws.participantnum).flat[0])
    series = _series_id_from_filename(path.name)

    trials = []
    for i, w in enumerate(wins):
        trial = {
            "eeg":              _to_f32(w.eeg),
            "emg":              _to_f32(w.emg),
            "kin":              _to_f32(w.kin),
            "eeg_t":            _to_f32(w.eeg_t),
            "emg_t":            _to_f32(w.emg_t),
            "weight":           int(_scalar(w.weight)),
            "surf":             int(_scalar(w.surf)),
            "weight_id":        str(w.weight_id).strip(),
            "surf_id":          str(w.surf_id).strip(),
            "led_on":           _scalar(w.LEDon),
            "led_off":          _scalar(w.LEDoff),
            "trial_start_time": _scalar(w.trial_start_time),
            "trial_idx":        i,
            "participant":      participant,
            "series":           series,
            "path":             str(path),
        }
        trials.append(trial)

    return trials


# ---------------------------------------------------------------------------
# Participant-level loader
# ---------------------------------------------------------------------------

def load_participant(
    data_dir: Union[str, Path],
    participant: int,
    file_type: str = "hs",
    series: list | None = None,
    include_stability: bool = True,
) -> list[dict]:
    """Load all requested series for one participant.

    Args:
        data_dir:          Root directory containing P1/, P2/, ... subdirs.
        participant:       Participant number 1-12.
        file_type:         'hs' (continuous) or 'ws' (per-trial windowed).
        series:            List of series IDs to load (ints and/or 'ST').
                           None = load all available.
        include_stability: When series=None, include the ST series.

    Returns:
        List of series dicts (load_hs) or lists of trial dicts (load_ws),
        sorted by series number with ST appended last.
    """
    data_dir = Path(data_dir)
    p_dir = data_dir / f"P{participant}"
    if not p_dir.exists():
        raise FileNotFoundError(f"Participant directory not found: {p_dir}")

    prefix = file_type.upper()
    found_files = sorted(p_dir.glob(f"{prefix}_P{participant}_S*.mat"))

    if not found_files:
        raise FileNotFoundError(
            f"No {prefix} files found for P{participant} in {p_dir}"
        )

    series_map: dict[Union[int, str], Path] = {}
    for f in found_files:
        try:
            sid = _series_id_from_filename(f.name)
        except ValueError:
            continue
        series_map[sid] = f

    if series is not None:
        target_series = series
    else:
        numeric = sorted(k for k in series_map if isinstance(k, int))
        target_series = numeric
        if include_stability and "ST" in series_map:
            target_series = target_series + ["ST"]

    loader_fn = load_hs if file_type == "hs" else load_ws

    results = []
    for sid in target_series:
        if sid not in series_map:
            continue
        results.append(loader_fn(series_map[sid]))

    return results

In [ ]:
"""
PyTorch Dataset for WAY-EEG-GAL.

Loads continuous HS series, extracts the 13-dim kinematic state vector k_t,
slides a fixed-length window over each series, and returns
(eeg_window, kin_window, emg_window) tensors.

Kinematic column mapping in hs.kin.sig (36 cols, 0-indexed):
  0-2   Px1,Py1,Pz1  object position (mm)
  3-5   Px2,Py2,Pz2  index fingertip position (mm)
  6-8   Px3,Py3,Pz3  thumb position (mm)
  9-11  Px4,Py4,Pz4  wrist position (mm)
  12-14 FX1,FY1,FZ1  force plate 1 (index), N
  15-17 FX2,FY2,FZ2  force plate 2 (thumb), N
  18-23 TX1..TZ2     torques, N*mm
  24-35 misc env / remaining

k_t construction (13-dim):
  p_wrist (3)  = cols 9:12
  p_index (3)  = cols 3:6
  p_thumb (3)  = cols 6:9
  d_grip  (1)  = ||p_index - p_thumb||_2
  F_L     (1)  = col 12  (FX1, load force index)
  F_G     (1)  = abs(col 15)  (FZ1 negated in hardware)
  rho_GL  (1)  = F_G / (F_L + eps)
"""

from __future__ import annotations

from pathlib import Path
from typing import Callable, Union

import numpy as np
import torch
from torch import Tensor
from torch.utils.data import Dataset


# Column indices in hs.kin.sig (0-based)
_KIN_WRIST = slice(9, 12)    # Px4, Py4, Pz4
_KIN_INDEX = slice(3, 6)     # Px2, Py2, Pz2
_KIN_THUMB = slice(6, 9)     # Px3, Py3, Pz3
_KIN_FX1   = 12              # load force index plate  (N)
_KIN_FZ1   = 15              # grip force index plate  (N, may be negated)

_EPS = 1e-8


def extract_kt(kin: np.ndarray) -> np.ndarray:
    """Build 13-dim kinematic state vector from raw 36-channel kin array.

    Args:
        kin: ndarray (T, 36)

    Returns:
        ndarray (T, 13) = [p_wrist(3), p_index(3), p_thumb(3),
                           d_grip(1), F_L(1), F_G(1), rho_GL(1)]
    """
    p_wrist = kin[:, _KIN_WRIST]               # (T, 3)
    p_index = kin[:, _KIN_INDEX]               # (T, 3)
    p_thumb = kin[:, _KIN_THUMB]               # (T, 3)

    d_grip = np.linalg.norm(p_index - p_thumb, axis=1, keepdims=True)  # (T, 1)

    F_L = kin[:, _KIN_FX1: _KIN_FX1 + 1]      # (T, 1)  load force
    F_G = np.abs(kin[:, _KIN_FZ1: _KIN_FZ1 + 1])  # (T, 1) grip force (abs)
    rho_GL = F_G / (np.abs(F_L) + _EPS)        # (T, 1)

    return np.concatenate(
        [p_wrist, p_index, p_thumb, d_grip, F_L, F_G, rho_GL], axis=1
    ).astype(np.float32)


class WAYEEGDataset(Dataset):
    """Sliding-window dataset over WAY-EEG-GAL HS series.

    Each sample is a tuple (eeg, kin, emg) of fixed-length windows:
        eeg : Tensor (window_size, 32)
        kin : Tensor (window_size, 13)   k_t kinematic state
        emg : Tensor (window_size, 5)    EMG envelope target

    EMG is at 4000 Hz in the raw files. Before windowing it must be
    downsampled to 500 Hz (8x) so that EEG, kin, and EMG share the
    same time axis. Pass a `preprocess_fn` that handles this
    (and any other preprocessing) before windowing.

    Args:
        data_dir:       Root dir with P1/, P2/, ... subdirs.
        participants:   List of participant IDs (1-12).
        split:          'train' | 'val' | 'test' | 'stability' | 'all'.
        window_size:    Number of samples per window (default 500 = 1 s @ 500 Hz).
        stride:         Step between consecutive windows (default 50 = 100 ms).
        preprocess_fn:  Optional callable (series_dict) -> series_dict applied
                        to each raw series before windowing. Must ensure
                        emg shape is (T, 5) at fs_eeg samples/s.
        cache_dir:      Optional directory to cache processed .npz files.
                        Filename: cache_dir/P{p}_{split}_S{s}.npz
    """

    def __init__(
        self,
        data_dir: Union[str, Path],
        participants: list[int],
        split: str,
        window_size: int = 500,
        stride: int = 50,
        preprocess_fn: Callable | None = None,
        cache_dir: Union[str, Path, None] = None,
    ) -> None:
        self.data_dir    = Path(data_dir)
        self.participants = participants
        self.split       = split
        self.window_size = window_size
        self.stride      = stride
        self.preprocess_fn = preprocess_fn
        self.cache_dir   = Path(cache_dir) if cache_dir else None

        # Index: list of (eeg_array, kin_array, emg_array, start_sample)
        # Built lazily in _build_index
        self._windows: list[tuple[np.ndarray, np.ndarray, np.ndarray, int]] = []
        self._build_index()
        # Drop the preprocessing closure after indexing: it is only needed
        # during _build_index and is never called from __getitem__.
        # Keeping it would make the dataset unpicklable under Windows
        # multiprocessing (spawn), breaking DataLoader with num_workers > 0.
        self.preprocess_fn = None

    # ------------------------------------------------------------------
    # Index construction
    # ------------------------------------------------------------------

    def _build_index(self) -> None:
        target_series = get_split_series(self.split)

        for p in self.participants:
            try:
                series_list = load_participant(
                    self.data_dir,
                    participant=p,
                    file_type="hs",
                    series=target_series,
                    include_stability=("ST" in target_series),
                )
            except FileNotFoundError:
                continue

            for s in series_list:
                self._index_series(s, p)

    def _index_series(self, series: dict, participant: int) -> None:
        """Slice one series into windows and append to self._windows."""
        cache_key = f"P{participant}_{self.split}_S{series['series']}"

        # Try cache first
        if self.cache_dir is not None:
            cached = self._load_cache(cache_key)
            if cached is not None:
                eeg_all, kin_all, emg_all = cached
                self._slide_windows(eeg_all, kin_all, emg_all)
                return

        # Apply preprocessing if supplied
        if self.preprocess_fn is not None:
            series = self.preprocess_fn(series)

        eeg = series["eeg"]   # (T, 32)
        kin = series["kin"]   # (T, 36)
        emg = series["emg"]   # (T, 5)  must be at same fs as eeg after preprocess

        kt = extract_kt(kin)  # (T, 13)

        # Trim all to same length (in case preprocess left minor length diff)
        T = min(eeg.shape[0], kt.shape[0], emg.shape[0])
        eeg, kt, emg = eeg[:T], kt[:T], emg[:T]

        if self.cache_dir is not None:
            self._save_cache(cache_key, eeg, kt, emg)

        self._slide_windows(eeg, kt, emg)

    def _slide_windows(
        self,
        eeg: np.ndarray,
        kt:  np.ndarray,
        emg: np.ndarray,
    ) -> None:
        T = eeg.shape[0]
        W = self.window_size
        S = self.stride
        for start in range(0, T - W + 1, S):
            self._windows.append((eeg, kt, emg, start))

    # ------------------------------------------------------------------
    # Cache helpers
    # ------------------------------------------------------------------

    def _cache_path(self, key: str) -> Path:
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        return self.cache_dir / f"{key}.npz"

    def _load_cache(self, key: str) -> tuple[np.ndarray, np.ndarray, np.ndarray] | None:
        p = self._cache_path(key)
        if not p.exists():
            return None
        data = np.load(p)
        return data["eeg"], data["kin"], data["emg"]

    def _save_cache(
        self,
        key: str,
        eeg: np.ndarray,
        kt:  np.ndarray,
        emg: np.ndarray,
    ) -> None:
        np.savez_compressed(self._cache_path(key), eeg=eeg, kin=kt, emg=emg)

    # ------------------------------------------------------------------
    # Dataset interface
    # ------------------------------------------------------------------

    def __len__(self) -> int:
        return len(self._windows)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor, Tensor]:
        eeg_all, kt_all, emg_all, start = self._windows[idx]
        end = start + self.window_size

        eeg_w = torch.from_numpy(eeg_all[start:end])   # (W, 32)
        kin_w = torch.from_numpy(kt_all[start:end])    # (W, 13)
        emg_w = torch.from_numpy(emg_all[start:end])   # (W, 5)

        return eeg_w, kin_w, emg_w

    # ------------------------------------------------------------------
    # Convenience
    # ------------------------------------------------------------------

    def __repr__(self) -> str:
        return (
            f"WAYEEGDataset("
            f"split={self.split!r}, "
            f"participants={self.participants}, "
            f"n_windows={len(self)}, "
            f"window_size={self.window_size}, "
            f"stride={self.stride})"
        )

In [ ]:
"""
EEG preprocessing pipeline for WAY-EEG-GAL HS series.

Pipeline (per method1.tex):
  1. Bandpass  0.1–40 Hz   4th-order zero-phase Butterworth
  2. Notch     50 Hz       power-line removal
  3. ASR                   sliding-window covariance-based artifact rejection
  4. CAR                   common average reference
  5. Delta     0.1–2 Hz    4th-order zero-phase Butterworth

Input:  raw EEG ndarray (T, 32) from load_hs(), float32, µV
Output: delta-band EEG ndarray (T, 32), float32
"""

from __future__ import annotations

import numpy as np
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt


# ---------------------------------------------------------------------------
# Individual steps
# ---------------------------------------------------------------------------

def bandpass(
    eeg: np.ndarray,
    fs: float,
    low: float = 0.1,
    high: float = 40.0,
    order: int = 4,
) -> np.ndarray:
    """Zero-phase Butterworth bandpass filter along time axis (axis=0)."""
    nyq = fs / 2.0
    sos = butter(order, [low / nyq, high / nyq], btype="bandpass", output="sos")
    return sosfiltfilt(sos, eeg, axis=0).astype(np.float32)


def notch(
    eeg: np.ndarray,
    fs: float,
    freq: float = 50.0,
    quality: float = 30.0,
) -> np.ndarray:
    """IIR notch filter for power-line removal."""
    b, a = iirnotch(freq, quality, fs)
    return filtfilt(b, a, eeg, axis=0).astype(np.float32)


def asr(
    eeg: np.ndarray,
    fs: float,
    window_ms: float = 500.0,
    std_thresh: float = 5.0,
    baseline_sec: float = 30.0,
) -> np.ndarray:
    """Artifact Subspace Reconstruction (simplified sliding-window version).

    Algorithm:
      1. Estimate per-channel robust std from the first `baseline_sec` seconds
         (assumed cleaner than the rest of the recording).
      2. Slide a window of `window_ms` ms over the signal.
      3. For each window, compute the per-channel std.
      4. Channels whose std exceeds `std_thresh` × baseline_std in that window
         are replaced by the channel mean of the non-artifact channels within
         the window (linear interpolation in the channel domain).
      5. Samples within windows where *all* channels exceed the threshold are
         zeroed (no clean reference available).

    This is a lightweight approximation of the full Riemannian ASR used in
    EEGLAB. It preserves the signal shape while suppressing high-amplitude
    transients without requiring the full matrix-decomposition loop.

    Args:
        eeg:          ndarray (T, C) float32
        fs:           sampling rate (Hz)
        window_ms:    sliding window length (ms)
        std_thresh:   rejection threshold in multiples of baseline std
        baseline_sec: seconds at start of recording used to estimate baseline

    Returns:
        ndarray (T, C) float32 with artifacts attenuated
    """
    T, C = eeg.shape
    win_samples = max(1, int(round(window_ms * fs / 1000.0)))
    baseline_samples = min(T, int(round(baseline_sec * fs)))

    # Robust baseline std (median absolute deviation scaled to std)
    baseline = eeg[:baseline_samples]
    baseline_med = np.median(baseline, axis=0, keepdims=True)
    baseline_std = np.median(np.abs(baseline - baseline_med), axis=0) / 0.6745
    baseline_std = np.maximum(baseline_std, 1e-6)   # avoid division by zero

    threshold = std_thresh * baseline_std            # (C,)

    out = eeg.copy()

    for start in range(0, T - win_samples + 1, win_samples):
        end = start + win_samples
        win = out[start:end]                         # (W, C)
        win_std = win.std(axis=0)                    # (C,)

        bad_mask = win_std > threshold               # (C,) bool
        good_mask = ~bad_mask

        if bad_mask.any():
            if good_mask.any():
                # Replace bad channels with mean of good channels (per sample)
                good_mean = win[:, good_mask].mean(axis=1, keepdims=True)
                out[start:end, bad_mask] = np.broadcast_to(
                    good_mean, (win_samples, bad_mask.sum())
                )
            else:
                # All channels bad — zero out window
                out[start:end] = 0.0

    return out.astype(np.float32)


def common_average_reference(eeg: np.ndarray) -> np.ndarray:
    """Subtract mean across all channels at each time point (CAR)."""
    return (eeg - eeg.mean(axis=1, keepdims=True)).astype(np.float32)


def delta_band(
    eeg: np.ndarray,
    fs: float,
    low: float = 0.1,
    high: float = 2.0,
    order: int = 4,
) -> np.ndarray:
    """Zero-phase Butterworth bandpass for delta band (0.1–2 Hz)."""
    nyq = fs / 2.0
    sos = butter(order, [low / nyq, high / nyq], btype="bandpass", output="sos")
    return sosfiltfilt(sos, eeg, axis=0).astype(np.float32)


def select_channels(
    eeg: np.ndarray,
    channel_names: list[str],
    target_channels: list[str] | None = None,
) -> tuple[np.ndarray, list[str]]:
    """Select a specific subset of EEG channels.

    Args:
        eeg: ndarray (T, C) float32
        channel_names: list of C channel strings
        target_channels: list of channels to keep. Defaults to:
                         FC1, FC2, FC3, FC4, C3, C4, CZ, CP1, CP2, CP3, CP4, CP5, CP6, CPZ

    Returns:
        (filtered_eeg, filtered_channel_names)
    """
    if target_channels is None:
        return eeg, channel_names


    # Use case-insensitive matching in case dataset names are e.g., "Cz" instead of "CZ"
    names_lower = [str(n).lower().strip() for n in channel_names]
    indices = []
    found_names = []

    for ch in target_channels:
        ch_lower = ch.lower().strip()
        if ch_lower in names_lower:
            idx = names_lower.index(ch_lower)
            indices.append(idx)
            found_names.append(channel_names[idx])
        else:
            raise ValueError(f"Target channel '{ch}' not found in provided channel_names: {channel_names}")

    return eeg[:, indices].astype(np.float32), found_names


# ---------------------------------------------------------------------------
# Full pipeline
# ---------------------------------------------------------------------------

def preprocess_eeg(
    eeg: np.ndarray,
    fs: float = 500.0,
    bp_low: float = 0.1,
    bp_high: float = 40.0,
    filter_order: int = 4,
    notch_freq: float = 50.0,
    notch_quality: float = 30.0,
    asr_window_ms: float = 500.0,
    asr_std_thresh: float = 5.0,
    asr_baseline_sec: float = 30.0,
    delta_low: float = 0.1,
    delta_high: float = 2.0,
    channel_names: list[str] | None = None,
    target_channels: list[str] | None = None,
) -> np.ndarray:
    """Apply full EEG preprocessing pipeline to one continuous HS series.

    Steps (method1.tex order):
      Channel selection (optional) → BP 0.1–40 Hz → notch 50 Hz → ASR → CAR → delta 0.1–2 Hz

    Args:
        eeg: ndarray (T, 32) float32, raw µV from load_hs()
        fs:  EEG sampling rate (default 500 Hz)
        ... (see individual step parameters above)

    Returns:
        ndarray (T, C) float32
        Delta-band EEG ready for CCA / windowing.
    """
    if channel_names is not None:
        eeg, channel_names = select_channels(eeg, channel_names, target_channels)

    eeg = bandpass(eeg, fs, bp_low, bp_high, filter_order)
    eeg = notch(eeg, fs, notch_freq, notch_quality)
    eeg = asr(eeg, fs, asr_window_ms, asr_std_thresh, asr_baseline_sec)
    eeg = common_average_reference(eeg)
    eeg = delta_band(eeg, fs, delta_low, delta_high, filter_order)
    return eeg


def preprocess_eeg_from_config(
    eeg: np.ndarray, fs: float, cfg: dict, channel_names: list[str] | None = None
) -> np.ndarray:
    """Convenience wrapper accepting a config dict (preprocessing.eeg section).

    Example cfg:
        {bp_low: 0.1, bp_high: 40.0, filter_order: 4,
         notch_freq: 50.0, asr_window_ms: 500, asr_std_thresh: 5.0,
         delta_low: 0.1, delta_high: 2.0}
    """
    return preprocess_eeg(
        eeg,
        fs=fs,
        bp_low=cfg.get("bp_low", 0.1),
        bp_high=cfg.get("bp_high", 40.0),
        filter_order=cfg.get("filter_order", 4),
        notch_freq=cfg.get("notch_freq", 50.0),
        notch_quality=cfg.get("notch_quality", 30.0),
        asr_window_ms=cfg.get("asr_window_ms", 500.0),
        asr_std_thresh=cfg.get("asr_std_thresh", 5.0),
        asr_baseline_sec=cfg.get("asr_baseline_sec", 30.0),
        delta_low=cfg.get("delta_low", 0.1),
        delta_high=cfg.get("delta_high", 2.0),
        channel_names=channel_names,
        target_channels=cfg.get("target_channels"),
    )

In [ ]:
"""
EMG preprocessing pipeline for WAY-EEG-GAL HS series.

Pipeline (per method1.tex):
  1. Bandpass  30–300 Hz   4th-order Butterworth
  2. Rectify   |s(t)|      full-wave
  3. Low-pass  10 Hz       smooth activation envelope
  4. Decimate  ×8          4000 → 500 Hz (matches EEG/kin sampling rate)
  5. Z-score               per channel (fit statistics on train set only)

Input:  raw EMG ndarray (T_emg, 5) from load_hs() at 4000 Hz
Output: envelope ndarray (T_eeg, 5) at 500 Hz, float32
"""

from __future__ import annotations

import numpy as np
from scipy.signal import butter, sosfiltfilt, decimate


# ---------------------------------------------------------------------------
# Individual steps
# ---------------------------------------------------------------------------

def bandpass(
    emg: np.ndarray,
    fs: float,
    low: float = 30.0,
    high: float = 300.0,
    order: int = 4,
) -> np.ndarray:
    """Zero-phase Butterworth bandpass filter (axis=0)."""
    nyq = fs / 2.0
    sos = butter(order, [low / nyq, high / nyq], btype="bandpass", output="sos")
    return sosfiltfilt(sos, emg, axis=0).astype(np.float32)


def rectify(emg: np.ndarray) -> np.ndarray:
    """Full-wave rectification."""
    return np.abs(emg).astype(np.float32)


def lowpass_envelope(
    emg: np.ndarray,
    fs: float,
    cutoff: float = 10.0,
    order: int = 4,
) -> np.ndarray:
    """Zero-phase Butterworth low-pass to extract smooth activation envelope."""
    nyq = fs / 2.0
    sos = butter(order, cutoff / nyq, btype="low", output="sos")
    return sosfiltfilt(sos, emg, axis=0).astype(np.float32)


def downsample(
    emg: np.ndarray,
    factor: int = 8,
) -> np.ndarray:
    """Decimate EMG by integer factor (4000 → 500 Hz with factor=8).

    Uses scipy.signal.decimate which applies an anti-aliasing filter before
    down-sampling. Applied per-channel to avoid across-channel filtering.
    """
    out = np.stack(
        [decimate(emg[:, c], factor, zero_phase=True) for c in range(emg.shape[1])],
        axis=1,
    )
    return out.astype(np.float32)


# ---------------------------------------------------------------------------
# Z-score normaliser (stateful — must be fit on train, applied to all splits)
# ---------------------------------------------------------------------------

class EMGNormalizer:
    """Per-channel z-score normalisation.

    Fit on the concatenated training-set EMG envelopes, then transform
    any split to zero-mean unit-variance.

    Usage:
        norm = EMGNormalizer()
        norm.fit(train_emg_list)     # list of (T, 5) arrays
        val_emg = norm.transform(val_emg)
    """

    def __init__(self) -> None:
        self.mean_: np.ndarray | None = None   # (5,)
        self.std_:  np.ndarray | None = None   # (5,)

    def fit(self, arrays: list[np.ndarray]) -> "EMGNormalizer":
        """Compute mean and std from a list of (T, C) arrays."""
        concat = np.concatenate(arrays, axis=0)   # (T_total, C)
        self.mean_ = concat.mean(axis=0).astype(np.float32)
        self.std_  = concat.std(axis=0).astype(np.float32)
        self.std_  = np.maximum(self.std_, 1e-6)
        return self

    def transform(self, emg: np.ndarray) -> np.ndarray:
        if self.mean_ is None:
            raise RuntimeError("EMGNormalizer must be fit before transform.")
        return ((emg - self.mean_) / self.std_).astype(np.float32)

    def fit_transform(self, arrays: list[np.ndarray]) -> list[np.ndarray]:
        self.fit(arrays)
        return [self.transform(a) for a in arrays]

    def state_dict(self) -> dict:
        return {"mean": self.mean_, "std": self.std_}

    def load_state_dict(self, d: dict) -> None:
        self.mean_ = np.asarray(d["mean"], dtype=np.float32)
        self.std_  = np.asarray(d["std"],  dtype=np.float32)


# ---------------------------------------------------------------------------
# Full pipeline (single series, no z-score — normaliser is separate)
# ---------------------------------------------------------------------------

def preprocess_emg(
    emg: np.ndarray,
    fs: float = 4000.0,
    bp_low: float = 30.0,
    bp_high: float = 300.0,
    filter_order: int = 4,
    lp_cutoff: float = 10.0,
    downsample_factor: int = 8,
) -> np.ndarray:
    """Apply EMG envelope extraction pipeline to one continuous HS series.

    Steps (method1.tex order):
      BP 30–300 Hz → |s(t)| → LP 10 Hz → decimate ×8

    Z-score normalisation is NOT applied here because it requires statistics
    computed across the whole training set. Use EMGNormalizer separately.

    Args:
        emg:              ndarray (T, 5) float32 at `fs` Hz from load_hs()
        fs:               EMG sampling rate (default 4000 Hz)
        bp_low/bp_high:   bandpass bounds (Hz)
        filter_order:     Butterworth filter order
        lp_cutoff:        envelope low-pass cutoff (Hz)
        downsample_factor: integer decimation factor (4000/500 = 8)

    Returns:
        ndarray (T // downsample_factor, 5) float32
        at fs // downsample_factor Hz (500 Hz), envelope only (no z-score)
    """
    emg = bandpass(emg, fs, bp_low, bp_high, filter_order)
    emg = rectify(emg)
    emg = lowpass_envelope(emg, fs, lp_cutoff, filter_order)
    emg = downsample(emg, downsample_factor)
    return emg


def preprocess_emg_from_config(emg: np.ndarray, fs: float, cfg: dict) -> np.ndarray:
    """Convenience wrapper accepting a config dict (preprocessing.emg section)."""
    return preprocess_emg(
        emg,
        fs=fs,
        bp_low=cfg.get("bp_low", 30.0),
        bp_high=cfg.get("bp_high", 300.0),
        filter_order=cfg.get("filter_order", 4),
        lp_cutoff=cfg.get("lp_cutoff", 10.0),
        downsample_factor=cfg.get("downsample_factor", 8),
    )

In [ ]:
"""
Kinematics preprocessing pipeline for WAY-EEG-GAL HS series.

Pipeline (per method1.tex):
  1. Velocity estimation        two methods (select via `velocity_method`):
       'sg'   — Savitzky-Golay first derivative (edge-preserving, default)
       'bw'   — Butterworth zero-phase differentiator (smoother, frequency-domain)
  2. Per-feature z-score        fit on train, apply to val/test
  3. k_t construction           13-dim state vector (same as dataset.py extract_kt)

Input:  raw kin ndarray (T, 36) from load_hs() at 500 Hz
Output: normalised k_t ndarray (T, 13) or (T, 26) with velocity, float32

Note: extract_kt() in dataset.py builds the raw 13-dim vector from the 36-col
signal. This module adds velocity features and per-feature normalisation.

If you only need the raw k_t vector without normalisation or velocity, use
`src/data/dataset.extract_kt` directly.
"""

from __future__ import annotations

import numpy as np
from scipy.signal import savgol_filter, butter, sosfiltfilt


# ---------------------------------------------------------------------------
# Column indices in raw 36-col kin signal (0-indexed)
# ---------------------------------------------------------------------------

_KIN_WRIST = slice(9, 12)      # Px4, Py4, Pz4 — wrist position  (mm)
_KIN_INDEX = slice(3, 6)       # Px2, Py2, Pz2 — index fingertip (mm)
_KIN_THUMB = slice(6, 9)       # Px3, Py3, Pz3 — thumb tip       (mm)
_KIN_FX1   = 12                # load force index plate           (N)
_KIN_FZ1   = 15                # grip force index plate           (N)
_EPS = 1e-8


# ---------------------------------------------------------------------------
# Velocity methods
# ---------------------------------------------------------------------------

def sg_velocity(
    pos: np.ndarray,
    fs: float = 500.0,
    window: int = 11,
    poly: int = 3,
) -> np.ndarray:
    """Velocity via Savitzky-Golay first derivative.

    Edge-preserving — good for sharp onset/offset of grasp kinematics.

    Args:
        pos:    (T, C) or (T,) position signal
        fs:     sampling rate (Hz) — scales output to mm/s
        window: SG window length (must be odd; auto-corrected if even)
        poly:   SG polynomial order

    Returns:
        velocity ndarray same shape as pos
    """
    if window % 2 == 0:
        window += 1
    return savgol_filter(pos, window_length=window, polyorder=poly,
                         deriv=1, delta=1.0 / fs, axis=0).astype(np.float32)


def bw_velocity(
    pos: np.ndarray,
    fs: float = 500.0,
    cutoff: float = 20.0,
    order: int = 4,
) -> np.ndarray:
    """Velocity via Butterworth zero-phase differentiator.

    Smoothest in the frequency domain — suppresses high-freq noise before
    numerical differentiation. Better when position signal has residual HF
    noise after filtering.

    Steps:
      1. Zero-phase Butterworth LP at `cutoff` Hz  (remove HF noise)
      2. Central finite difference  Δpos / Δt      (numerical derivative)

    Args:
        pos:    (T, C) or (T,) position signal
        fs:     sampling rate (Hz)
        cutoff: LP cutoff before differentiation (Hz)
        order:  Butterworth filter order

    Returns:
        velocity ndarray same shape as pos
    """
    nyq = fs / 2.0
    sos = butter(order, cutoff / nyq, btype="low", output="sos")
    pos_smooth = sosfiltfilt(sos, pos, axis=0)
    # Central difference; pad endpoints with one-sided difference
    vel = np.gradient(pos_smooth, 1.0 / fs, axis=0)
    return vel.astype(np.float32)


def estimate_velocity(
    pos: np.ndarray,
    fs: float = 500.0,
    method: str = "sg",
    sg_window: int = 11,
    sg_poly: int = 3,
    bw_cutoff: float = 20.0,
    bw_order: int = 4,
) -> np.ndarray:
    """Unified velocity estimator — dispatch to sg or bw method.

    Args:
        pos:       (T, C) or (T,) position signal
        fs:        sampling rate (Hz)
        method:    'sg' (Savitzky-Golay) | 'bw' (Butterworth differentiator)
        sg_window: SG window length (used when method='sg')
        sg_poly:   SG polynomial order (used when method='sg')
        bw_cutoff: LP cutoff Hz before diff (used when method='bw')
        bw_order:  Butterworth order (used when method='bw')

    Returns:
        velocity ndarray same shape as pos
    """
    if method == "sg":
        return sg_velocity(pos, fs=fs, window=sg_window, poly=sg_poly)
    if method == "bw":
        return bw_velocity(pos, fs=fs, cutoff=bw_cutoff, order=bw_order)
    raise ValueError(f"Unknown velocity method '{method}'. Choose 'sg' or 'bw'.")


# ---------------------------------------------------------------------------
# k_t extraction (raw, no normalisation)
# ---------------------------------------------------------------------------

def extract_kt_raw(kin: np.ndarray) -> np.ndarray:
    """Build 13-dim kinematic state vector from raw 36-col kin signal.

    Mirrors dataset.extract_kt but lives here for pipeline use.

    Returns:
        (T, 13): [p_wrist(3), p_index(3), p_thumb(3), d_grip(1), F_L(1), F_G(1), rho_GL(1)]
    """
    p_wrist = kin[:, _KIN_WRIST]                                 # (T, 3)
    p_index = kin[:, _KIN_INDEX]                                 # (T, 3)
    p_thumb = kin[:, _KIN_THUMB]                                 # (T, 3)
    d_grip  = np.linalg.norm(p_index - p_thumb, axis=1, keepdims=True)  # (T, 1)
    F_L     = kin[:, _KIN_FX1: _KIN_FX1 + 1]                   # (T, 1)
    F_G     = np.abs(kin[:, _KIN_FZ1: _KIN_FZ1 + 1])            # (T, 1)
    rho_GL  = F_G / (np.abs(F_L) + _EPS)                        # (T, 1)
    return np.concatenate(
        [p_wrist, p_index, p_thumb, d_grip, F_L, F_G, rho_GL], axis=1
    ).astype(np.float32)


# ---------------------------------------------------------------------------
# Velocity-augmented k_t (optional, 26-dim)
# ---------------------------------------------------------------------------

def extract_kt_with_velocity(
    kin: np.ndarray,
    fs: float = 500.0,
    method: str = "sg",
    sg_window: int = 11,
    sg_poly: int = 3,
    bw_cutoff: float = 20.0,
    bw_order: int = 4,
) -> np.ndarray:
    """Build 26-dim k_t with appended velocity features.

    Args:
        kin:       (T, 36) raw kinematic signal
        fs:        sampling rate (Hz)
        method:    'sg' or 'bw' — velocity estimation method
        sg_window: SG window (used when method='sg')
        sg_poly:   SG polynomial order (used when method='sg')
        bw_cutoff: LP cutoff Hz (used when method='bw')
        bw_order:  Butterworth order (used when method='bw')

    Returns:
        (T, 26): concat of raw k_t (13) + velocity of all 13 k_t dims
    """
    kt = extract_kt_raw(kin)                                    # (T, 13)
    vel = estimate_velocity(
        kt, fs=fs, method=method,
        sg_window=sg_window, sg_poly=sg_poly,
        bw_cutoff=bw_cutoff, bw_order=bw_order,
    )                                                           # (T, 13)
    return np.concatenate([kt, vel], axis=1).astype(np.float32)  # (T, 26)


# ---------------------------------------------------------------------------
# Normaliser (stateful, fit on train)
# ---------------------------------------------------------------------------

class KinNormalizer:
    """Per-feature z-score normalisation for k_t arrays.

    Fit on concatenated training-set k_t arrays, then apply to any split.

    Usage:
        norm = KinNormalizer()
        norm.fit(train_kt_list)
        val_kt = norm.transform(val_kt)
    """

    def __init__(self) -> None:
        self.mean_: np.ndarray | None = None
        self.std_:  np.ndarray | None = None

    def fit(self, arrays: list[np.ndarray]) -> "KinNormalizer":
        concat = np.concatenate(arrays, axis=0)
        self.mean_ = concat.mean(axis=0).astype(np.float32)
        self.std_  = concat.std(axis=0).astype(np.float32)
        self.std_  = np.maximum(self.std_, 1e-6)
        return self

    def transform(self, kt: np.ndarray) -> np.ndarray:
        if self.mean_ is None:
            raise RuntimeError("KinNormalizer must be fit before transform.")
        return ((kt - self.mean_) / self.std_).astype(np.float32)

    def fit_transform(self, arrays: list[np.ndarray]) -> list[np.ndarray]:
        self.fit(arrays)
        return [self.transform(a) for a in arrays]

    def state_dict(self) -> dict:
        return {"mean": self.mean_, "std": self.std_}

    def load_state_dict(self, d: dict) -> None:
        self.mean_ = np.asarray(d["mean"], dtype=np.float32)
        self.std_  = np.asarray(d["std"],  dtype=np.float32)


# ---------------------------------------------------------------------------
# Full pipeline (single series, no normalisation — normaliser is separate)
# ---------------------------------------------------------------------------

def preprocess_kinematics(
    kin: np.ndarray,
    fs: float = 500.0,
    include_velocity: bool = False,
    velocity_method: str = "sg",
    sg_window: int = 11,
    sg_poly: int = 3,
    bw_cutoff: float = 20.0,
    bw_order: int = 4,
) -> np.ndarray:
    """Extract k_t from raw 36-col kin signal, optionally with velocity.

    Args:
        kin:              ndarray (T, 36) from load_hs()
        fs:               kinematic sampling rate (default 500 Hz)
        include_velocity: if True return (T, 26); if False return (T, 13)
        velocity_method:  'sg' (Savitzky-Golay) | 'bw' (Butterworth diff)
        sg_window:        SG window length (used when velocity_method='sg')
        sg_poly:          SG polynomial order (used when velocity_method='sg')
        bw_cutoff:        LP cutoff Hz before diff (used when velocity_method='bw')
        bw_order:         Butterworth order (used when velocity_method='bw')

    Returns:
        ndarray (T, 13) or (T, 26) float32 — un-normalised k_t
        Use KinNormalizer to apply z-score over the training set.
    """
    if include_velocity:
        return extract_kt_with_velocity(
            kin, fs=fs, method=velocity_method,
            sg_window=sg_window, sg_poly=sg_poly,
            bw_cutoff=bw_cutoff, bw_order=bw_order,
        )
    return extract_kt_raw(kin)


def preprocess_kinematics_from_config(
    kin: np.ndarray,
    fs: float,
    cfg: dict,
) -> np.ndarray:
    """Convenience wrapper accepting a config dict (preprocessing.kinematics section)."""
    return preprocess_kinematics(
        kin,
        fs=fs,
        include_velocity=cfg.get("include_velocity", False),
        velocity_method=cfg.get("velocity_method", "sg"),
        sg_window=cfg.get("sg_window", 11),
        sg_poly=cfg.get("sg_poly", 3),
        bw_cutoff=cfg.get("bw_cutoff", 20.0),
        bw_order=cfg.get("bw_order", 4),
    )

In [ ]:
"""
CCA-based EEG alignment for KG-GT pipeline (method1.tex §3.2).

Goal: reduce EEG 32 → 16 canonical components maximally correlated
with kinematic state k_t (13-dim), so the Transformer sees neural
activity most relevant to motor execution.

Algorithm:
  1. Fit CCA on training-set (eeg, kin) pairs using sklearn CCA.
     - n_components = 16  (hardcoded in default.yaml)
     - eeg shape: (T_train, 32)   delta-band preprocessed
     - kin shape: (T_train, 13)   k_t from preprocess_kinematics
  2. Transform any split: project eeg (T, 32) → (T, 16) canonical scores.

Note: CCA requires both eeg and kin to fit, but only eeg to transform.
This is different from PCA/ICA — the projection is supervised by kinematics.
"""

from __future__ import annotations

import numpy as np
import torch
from torch import Tensor


class EEGKinCCA:
    """CCA alignment: EEG 32 → n_components canonical variates.

    Fit once on all training series. Then call transform(eeg) on any split
    to project EEG into the kinematically-aligned subspace.

    Usage:
        cca = EEGKinCCA(n_components=16)

        # Fit on train: provide lists of aligned (eeg, kin) arrays
        cca.fit(train_eeg_list, train_kin_list)

        # Transform any split
        eeg_aligned = cca.transform(val_eeg)     # (T, 16)
    """

    def __init__(self, n_components: int = 16, max_fit_samples: int = 500_000,
                 random_state: int = 42) -> None:
        self.n_components = n_components
        self.max_fit_samples = max_fit_samples
        self.random_state = random_state
        self._cca: CCA | None = None

    def fit(
        self,
        eeg_arrays: list[np.ndarray],
        kin_arrays: list[np.ndarray],
    ) -> "EEGKinCCA":
        """Fit CCA on concatenated training (eeg, kin) pairs.

        Args:
            eeg_arrays: list of (T_i, 32) preprocessed EEG arrays (train split)
            kin_arrays: list of (T_i, 13) k_t kinematic arrays (same split)

        All arrays must have the same number of samples T_i per pair.
        Arrays from different series are concatenated before fitting. If the
        total exceeds ``max_fit_samples``, a random subset is drawn — CCA
        weights converge on a fraction of the data and the full set (tens of
        millions of rows) would exhaust memory (sklearn copies X and y to
        float64 internally).
        """
        lengths = [e.shape[0] for e in eeg_arrays]
        for i, (n_e, k) in enumerate(zip(lengths, kin_arrays)):
            if n_e != k.shape[0]:
                raise ValueError(
                    f"EEG and kin sample counts differ in series {i}: "
                    f"{n_e} vs {k.shape[0]}"
                )

        n_total = int(np.sum(lengths))
        if self.max_fit_samples and n_total > self.max_fit_samples:
            # Sample per-array (proportional) and only materialise the subset,
            # so we never build the full (n_total, 32) concatenation in memory.
            rng = np.random.default_rng(self.random_state)
            eeg_parts, kin_parts = [], []
            for eeg_i, kin_i, n_i in zip(eeg_arrays, kin_arrays, lengths):
                take = max(1, round(self.max_fit_samples * n_i / n_total))
                take = min(take, n_i)
                sel = rng.choice(n_i, size=take, replace=False)
                sel.sort()
                eeg_parts.append(eeg_i[sel])
                kin_parts.append(kin_i[sel])
            eeg_concat = np.concatenate(eeg_parts, axis=0).astype(np.float64)
            kin_concat = np.concatenate(kin_parts, axis=0).astype(np.float64)
        else:
            eeg_concat = np.concatenate(eeg_arrays, axis=0).astype(np.float64)
            kin_concat = np.concatenate(kin_arrays, axis=0).astype(np.float64)

        # sklearn CCA (canonical mode) requires n_components <= min(n, p, q)
        max_components = min(eeg_concat.shape[0], eeg_concat.shape[1], kin_concat.shape[1])
        actual = min(self.n_components, max_components)
        if actual < self.n_components:
            import warnings
            warnings.warn(
                f"EEGKinCCA: n_components={self.n_components} exceeds upper bound "
                f"{max_components} (min of n_samples={eeg_concat.shape[0]}, "
                f"n_eeg={eeg_concat.shape[1]}, n_kin={kin_concat.shape[1]}). "
                f"Clamped to {actual}.",
                UserWarning, stacklevel=2,
            )
            self.n_components = actual

        self._cca = CCA(n_components=self.n_components, max_iter=1000)
        self._cca.fit(eeg_concat, kin_concat)
        return self

    def transform(self, eeg: np.ndarray) -> np.ndarray:
        """Project EEG into canonical space.

        Args:
            eeg: ndarray (T, 32) preprocessed EEG (any split)

        Returns:
            ndarray (T, n_components) float32 canonical EEG scores
        """
        if self._cca is None:
            raise RuntimeError("EEGKinCCA must be fit before transform.")
        # sklearn CCA.transform returns (X_scores, Y_scores); we only need X
        eeg_scores, _ = self._cca.transform(
            eeg.astype(np.float64),
            np.zeros((eeg.shape[0], 13), dtype=np.float64),
        )
        return eeg_scores.astype(np.float32)

    def fit_transform(
        self,
        eeg_arrays: list[np.ndarray],
        kin_arrays: list[np.ndarray],
    ) -> list[np.ndarray]:
        """Fit and transform all training arrays in one call.

        Returns:
            list of (T_i, n_components) arrays in same order as input
        """
        self.fit(eeg_arrays, kin_arrays)
        return [self.transform(e) for e in eeg_arrays]

    def state_dict(self) -> dict:
        """Serialise CCA weights for saving to .npz or checkpoint."""
        if self._cca is None:
            raise RuntimeError("EEGKinCCA has not been fit yet.")
        return {
            "n_components":  self.n_components,
            "x_weights_":    self._cca.x_weights_,     # (32, n_components)
            "y_weights_":    self._cca.y_weights_,     # (13, n_components)
            "x_mean_":       self._cca.x_mean_,        # (32,)
            "y_mean_":       self._cca.y_mean_,        # (13,)
            "x_std_":        self._cca.x_std_,         # (32,) or scalar
            "y_std_":        self._cca.y_std_,
        }

    def load_state_dict(self, d: dict) -> None:
        """Restore CCA weights without re-fitting."""
        self.n_components = int(d["n_components"])
        self._cca = CCA(n_components=self.n_components)
        # Manually restore sklearn internal attributes
        self._cca.x_weights_ = np.asarray(d["x_weights_"], dtype=np.float64)
        self._cca.y_weights_ = np.asarray(d["y_weights_"], dtype=np.float64)
        self._cca.x_mean_    = np.asarray(d["x_mean_"],    dtype=np.float64)
        self._cca.y_mean_    = np.asarray(d["y_mean_"],    dtype=np.float64)
        self._cca.x_std_     = np.asarray(d["x_std_"],     dtype=np.float64)
        self._cca.y_std_     = np.asarray(d["y_std_"],     dtype=np.float64)
        # Derive rotation matrices sklearn needs for transform()
        self._cca.x_rotations_ = self._cca.x_weights_
        self._cca.y_rotations_ = self._cca.y_weights_


    def torch_projector(self, device: torch.device | str = "cpu") -> "TorchCCA":
        """Export the fitted projection as an on-device TorchCCA.

        sklearn's CCA.transform reduces to an affine map followed by a matmul:
            X_scores = ((X - x_mean_) / x_std_) @ x_rotations_
        Replicating it in torch keeps the whole batch on the GPU and removes
        the per-batch CPU round-trip (.cpu().numpy() -> sklearn -> .to(device)).

        Args:
            device: Device to place the projection tensors on.

        Returns:
            TorchCCA with mean/std/rotation tensors on ``device``.
        """
        if self._cca is None:
            raise RuntimeError("EEGKinCCA must be fit before export.")

        # sklearn renamed these to private (_x_mean) around 1.3; support both.
        def _attr(*names):
            for n in names:
                if hasattr(self._cca, n):
                    return getattr(self._cca, n)
            raise AttributeError(f"CCA missing all of {names}")

        x_mean = np.asarray(_attr("_x_mean", "x_mean_"), dtype=np.float32).reshape(-1)
        x_std = np.asarray(_attr("_x_std", "x_std_"), dtype=np.float32).reshape(-1)
        x_rot = np.asarray(self._cca.x_rotations_, dtype=np.float32)  # (n_eeg, k)
        return TorchCCA(x_mean, x_std, x_rot, device=device)


class TorchCCA:
    """On-device EEG -> canonical projection (matmul only, no autograd needed).

    Mirrors EEGKinCCA.transform but runs entirely in torch so it can sit inside
    the GPU training loop. Not an nn.Module: the projection is a fixed,
    pre-fitted transform, so its tensors are plain buffers.
    """

    def __init__(
        self,
        x_mean: np.ndarray,
        x_std: np.ndarray,
        x_rotations: np.ndarray,
        device: torch.device | str = "cpu",
    ) -> None:
        self.device = torch.device(device)
        self.x_mean = torch.as_tensor(x_mean, dtype=torch.float32, device=self.device)
        self.x_std = torch.as_tensor(x_std, dtype=torch.float32, device=self.device)
        self.x_rotations = torch.as_tensor(
            x_rotations, dtype=torch.float32, device=self.device
        )
        self.n_components = self.x_rotations.shape[1]

    def to(self, device: torch.device | str) -> "TorchCCA":
        """Move projection tensors to ``device`` in place."""
        self.device = torch.device(device)
        self.x_mean = self.x_mean.to(self.device)
        self.x_std = self.x_std.to(self.device)
        self.x_rotations = self.x_rotations.to(self.device)
        return self

    @torch.no_grad()
    def transform(self, eeg: Tensor) -> Tensor:
        """Project EEG into canonical space on-device.

        Args:
            eeg: Tensor of shape (..., n_eeg) on any device.

        Returns:
            Tensor of shape (..., n_components) on this projector's device.
        """
        eeg = eeg.to(self.device, dtype=torch.float32)
        return ((eeg - self.x_mean) / self.x_std) @ self.x_rotations


# ---------------------------------------------------------------------------
# Config wrapper
# ---------------------------------------------------------------------------

def make_cca_from_config(cfg: dict) -> EEGKinCCA:
    """Build EEGKinCCA from preprocessing.cca config section."""
    return EEGKinCCA(
        n_components=cfg.get("n_components", 16),
        max_fit_samples=cfg.get("max_fit_samples", 500_000),
        random_state=cfg.get("random_state", 42),
    )

In [ ]:
"""
Transformer encoder for KG-GT (method1.tex §3.3.2).

Input is the CCA-aligned EEG sequence X̃ ∈ R^{B × T × d_in} (d_in = CCA
components, 13 by default). A linear embedding lifts it to d_model, sinusoidal
positional encoding is added (Eqs. 9-11), and L post-LN Transformer layers
model multi-lag cortico-muscular temporal coupling (Eqs. 12-17).

Output H_temp ∈ R^{B × T × d_model} feeds the kinematic-guided GAT.

Equation map (method1.tex):
  PE_sin / PE_cos          → SinusoidalPositionalEncoding   (Eqs. 9-10)
  Z^(0) = X̃ + PE           → TransformerEncoder.forward     (Eq. 11)
  Q,K,V / head / MHSA       → MultiHeadSelfAttention         (Eqs. 12-14)
  FFN                       → PositionwiseFeedForward        (Eq. 15)
  Z'^(l), Z^(l) (post-LN)   → TransformerEncoderLayer        (Eqs. 16-17)
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
from torch import Tensor


# ---------------------------------------------------------------------------
# Positional encoding — Eqs. 9-10
# ---------------------------------------------------------------------------

class SinusoidalPositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding added to the input sequence.

        PE[p, 2i]   = sin(p / 10000^(2i/d))
        PE[p, 2i+1] = cos(p / 10000^(2i/d))

    Precomputed up to max_len and registered as a (non-trainable) buffer so it
    moves with the module across devices and survives state_dict save/load.
    """

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.0) -> None:
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)                       # (max_len, d)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        # div_term[i] = 1 / 10000^(2i/d) = exp(-2i/d * ln(10000))
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))              # (1, max_len, d)

    def forward(self, x: Tensor) -> Tensor:
        """x: (B, T, d_model) → x + PE[:, :T]."""
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


# ---------------------------------------------------------------------------
# Multi-head self-attention — Eqs. 12-14
# ---------------------------------------------------------------------------

class MultiHeadSelfAttention(nn.Module):
    """Scaled dot-product multi-head self-attention.

        head_i = softmax(Q_i K_iᵀ / √d_k) V_i
        MHSA   = Concat(head_1..head_H) W_O

    Separate per-head projection dims (d_k, d_v) are supported, as in the
    config (d_k = d_v = 32, H = 8 ⇒ H·d_k = d_model = 256).
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_k: int,
        d_v: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_k
        self.d_v = d_v

        # Stacked projections for all heads at once: (d_model → H·d_*)
        self.W_Q = nn.Linear(d_model, n_heads * d_k, bias=False)
        self.W_K = nn.Linear(d_model, n_heads * d_k, bias=False)
        self.W_V = nn.Linear(d_model, n_heads * d_v, bias=False)
        self.W_O = nn.Linear(n_heads * d_v, d_model, bias=False)

        self.attn_dropout = nn.Dropout(p=dropout)
        self.scale = 1.0 / math.sqrt(d_k)

    def forward(self, x: Tensor) -> Tensor:
        """x: (B, T, d_model) → (B, T, d_model)."""
        B, T, _ = x.shape
        H, d_k, d_v = self.n_heads, self.d_k, self.d_v

        # Project then split into heads: (B, H, T, d_*)
        q = self.W_Q(x).view(B, T, H, d_k).transpose(1, 2)
        k = self.W_K(x).view(B, T, H, d_k).transpose(1, 2)
        v = self.W_V(x).view(B, T, H, d_v).transpose(1, 2)

        # Scaled dot-product attention (Eq. 13)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale   # (B,H,T,T)
        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        ctx = torch.matmul(attn, v)                                  # (B,H,T,d_v)

        # Concat heads and project (Eq. 14)
        ctx = ctx.transpose(1, 2).contiguous().view(B, T, H * d_v)
        return self.W_O(ctx)


# ---------------------------------------------------------------------------
# Position-wise feed-forward — Eq. 15
# ---------------------------------------------------------------------------

class PositionwiseFeedForward(nn.Module):
    """FFN(x) = max(0, x W_1 + b_1) W_2 + b_2  (ReLU)."""

    def __init__(self, d_model: int, ffn_dim: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.fc1 = nn.Linear(d_model, ffn_dim)
        self.fc2 = nn.Linear(ffn_dim, d_model)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: Tensor) -> Tensor:
        return self.fc2(self.dropout(self.act(self.fc1(x))))


# ---------------------------------------------------------------------------
# One encoder layer — post-LN, Eqs. 16-17
# ---------------------------------------------------------------------------

class TransformerEncoderLayer(nn.Module):
    """Post-LayerNorm Transformer encoder layer.

        Z'^(l) = LN( Z^(l-1) + MHSA(Z^(l-1)) )     (Eq. 16)
        Z^(l)  = LN( Z'^(l)  + FFN(Z'^(l))  )      (Eq. 17)

    Dropout is applied to each sublayer output before the residual add
    (standard "Attention Is All You Need" placement).
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_k: int,
        d_v: int,
        ffn_dim: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.mhsa = MultiHeadSelfAttention(d_model, n_heads, d_k, d_v, dropout)
        self.ffn = PositionwiseFeedForward(d_model, ffn_dim, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(p=dropout)
        self.dropout2 = nn.Dropout(p=dropout)

    def forward(self, x: Tensor) -> Tensor:
        # Sublayer 1: MHSA + residual + LN
        x = self.ln1(x + self.dropout1(self.mhsa(x)))
        # Sublayer 2: FFN + residual + LN
        x = self.ln2(x + self.dropout2(self.ffn(x)))
        return x


# ---------------------------------------------------------------------------
# Full encoder stack
# ---------------------------------------------------------------------------

class TransformerEncoder(nn.Module):
    """Stack of L post-LN encoder layers with input embedding + PE.

    Pipeline (method1.tex Eq. 11 onward):
        X̃ (B,T,d_in)
          → Linear embed (d_in → d_model)
          → + sinusoidal PE                       = Z^(0)
          → L × TransformerEncoderLayer
          → H_temp (B, T, d_model)

    The CCA stage outputs d_in (= n_components, 13) channels; the figure adds
    PE at that width but H_temp is 256-dim, so the embedding lifts d_in →
    d_model before the layers. If d_in == d_model the embedding is still a
    learnable linear map (no-op shape-wise).
    """

    def __init__(
        self,
        input_dim: int = 13,
        d_model: int = 256,
        n_layers: int = 4,
        n_heads: int = 8,
        d_k: int = 32,
        d_v: int = 32,
        ffn_dim: int = 1024,
        dropout: float = 0.2,
        max_len: int = 5000,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.d_model = d_model

        self.embed = nn.Linear(input_dim, d_model)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList(
            TransformerEncoderLayer(d_model, n_heads, d_k, d_v, ffn_dim, dropout)
            for _ in range(n_layers)
        )

    def forward(self, x: Tensor) -> Tensor:
        """x: CCA-aligned EEG (B, T, input_dim) → H_temp (B, T, d_model)."""
        if x.dim() != 3:
            raise ValueError(f"expected (B, T, input_dim), got {tuple(x.shape)}")
        if x.size(-1) != self.input_dim:
            raise ValueError(
                f"input feature dim {x.size(-1)} != input_dim {self.input_dim}"
            )

        z = self.embed(x)            # (B, T, d_model)
        z = self.pos_enc(z)          # + PE  → Z^(0)
        for layer in self.layers:
            z = layer(z)             # Z^(l)
        return z                     # H_temp


# ---------------------------------------------------------------------------
# Config wrapper
# ---------------------------------------------------------------------------

def build_transformer_from_config(cfg: dict, input_dim: int) -> TransformerEncoder:
    """Build TransformerEncoder from the model.transformer config section.

    Args:
        cfg: model.transformer dict (n_layers, n_heads, d_model, d_k, d_v,
             ffn_dim, dropout).
        input_dim: CCA output width (preprocessing.cca.n_components).
    """
    return TransformerEncoder(
        input_dim=input_dim,
        d_model=cfg.get("d_model", 256),
        n_layers=cfg.get("n_layers", 4),
        n_heads=cfg.get("n_heads", 8),
        d_k=cfg.get("d_k", 32),
        d_v=cfg.get("d_v", cfg.get("d_k", 32)),
        ffn_dim=cfg.get("ffn_dim", 1024),
        dropout=cfg.get("dropout", 0.2),
    )

In [ ]:
"""Projection from transformer temporal states to 5 muscle-node embeddings."""

from __future__ import annotations

import torch
import torch.nn as nn


class MuscleNodeProjection(nn.Module):
    """Project ``(B, T, D)`` transformer states into ``(B, T, N, F)`` nodes."""

    def __init__(self, input_dim: int, n_nodes: int = 5, node_dim: int = 32) -> None:
        super().__init__()
        self.n_nodes = n_nodes
        self.node_dim = node_dim
        self.proj = nn.Linear(input_dim, n_nodes * node_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() != 3:
            raise ValueError(f"expected (B, T, D), got {tuple(x.shape)}")
        batch_size, steps, _ = x.shape
        out = self.proj(x)
        return out.view(batch_size, steps, self.n_nodes, self.node_dim)

In [ ]:
"""Muscle-graph attention blocks for transformer-to-EMG decoding."""

from __future__ import annotations

import math

import torch
import torch.nn as nn


class MuscleGATLayer(nn.Module):
    """Timewise multi-head attention over the fixed 5-muscle graph."""

    def __init__(
        self,
        node_dim: int,
        hidden_dim: int,
        num_heads: int,
        dropout: float = 0.0,
        out_dim: int | None = None,
    ) -> None:
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.out_dim = out_dim or node_dim
        self.query = nn.Linear(node_dim, hidden_dim * num_heads, bias=False)
        self.key = nn.Linear(node_dim, hidden_dim * num_heads, bias=False)
        self.value = nn.Linear(node_dim, hidden_dim * num_heads, bias=False)
        self.out = nn.Linear(hidden_dim * num_heads, self.out_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(self.out_dim)
        self.residual = nn.Linear(node_dim, self.out_dim) if node_dim != self.out_dim else nn.Identity()
        self.scale = 1.0 / math.sqrt(hidden_dim)

    def forward(self, nodes: torch.Tensor) -> torch.Tensor:
        if nodes.dim() != 4:
            raise ValueError(f"expected (B, T, N, F), got {tuple(nodes.shape)}")
        batch_size, steps, n_nodes, _ = nodes.shape
        flat = nodes.reshape(batch_size * steps, n_nodes, -1)
        query = self.query(flat).view(batch_size * steps, n_nodes, self.num_heads, self.hidden_dim).permute(0, 2, 1, 3)
        key = self.key(flat).view(batch_size * steps, n_nodes, self.num_heads, self.hidden_dim).permute(0, 2, 1, 3)
        value = self.value(flat).view(batch_size * steps, n_nodes, self.num_heads, self.hidden_dim).permute(0, 2, 1, 3)

        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, value).permute(0, 2, 1, 3).reshape(batch_size * steps, n_nodes, self.num_heads * self.hidden_dim)
        out = self.out(out).view(batch_size, steps, n_nodes, self.out_dim)
        residual = self.residual(nodes)
        return self.norm(residual + self.dropout(out))


class MuscleGATEncoder(nn.Module):
    """Stacked baseline graph-attention encoder over the 5 EMG nodes."""

    def __init__(
        self,
        node_dim: int,
        hidden_dim: int,
        num_heads: int,
        out_dim: int,
        n_layers: int = 2,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        in_dim = node_dim
        for layer_idx in range(n_layers):
            layer_out = out_dim if layer_idx == n_layers - 1 else node_dim
            layers.append(
                MuscleGATLayer(
                    node_dim=in_dim,
                    hidden_dim=hidden_dim,
                    num_heads=num_heads,
                    dropout=dropout,
                    out_dim=layer_out,
                )
            )
            in_dim = layer_out
        self.layers = nn.ModuleList(layers)

    def forward(self, nodes: torch.Tensor) -> torch.Tensor:
        out = nodes
        for layer in self.layers:
            out = layer(out)
        return out


class KinematicGuidedMuscleGATEncoder(nn.Module):
    """Graph attention whose edge scores are conditioned on kinematics."""

    def __init__(
        self,
        node_dim: int,
        kin_dim: int,
        hidden_dim: int,
        num_heads: int,
        out_dim: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.node_proj = nn.Linear(node_dim, hidden_dim * num_heads, bias=False)
        self.kin_proj = nn.Linear(kin_dim, hidden_dim * num_heads, bias=False)
        self.value_proj = nn.Linear(node_dim, hidden_dim * num_heads, bias=False)
        self.out = nn.Linear(hidden_dim * num_heads, out_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(out_dim)
        self.residual = nn.Linear(node_dim, out_dim) if node_dim != out_dim else nn.Identity()
        self.scale = 1.0 / math.sqrt(hidden_dim)

    def forward(self, nodes: torch.Tensor, kin: torch.Tensor) -> torch.Tensor:
        if nodes.dim() != 4:
            raise ValueError(f"expected nodes (B, T, N, F), got {tuple(nodes.shape)}")
        if kin.dim() != 3:
            raise ValueError(f"expected kin (B, T, K), got {tuple(kin.shape)}")

        batch_size, steps, n_nodes, _ = nodes.shape
        flat_nodes = nodes.reshape(batch_size * steps, n_nodes, -1)
        flat_kin = kin.reshape(batch_size * steps, -1)

        node_proj = self.node_proj(flat_nodes).view(batch_size * steps, n_nodes, self.num_heads, self.hidden_dim).permute(0, 2, 1, 3)
        kin_proj = self.kin_proj(flat_kin).view(batch_size * steps, self.num_heads, self.hidden_dim).unsqueeze(2)
        values = self.value_proj(flat_nodes).view(batch_size * steps, n_nodes, self.num_heads, self.hidden_dim).permute(0, 2, 1, 3)

        guided = node_proj + kin_proj
        scores = torch.matmul(guided, guided.transpose(-2, -1)) * self.scale
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, values).permute(0, 2, 1, 3).reshape(batch_size * steps, n_nodes, self.num_heads * self.hidden_dim)
        out = self.out(out).view(batch_size, steps, n_nodes, -1)
        residual = self.residual(nodes)
        return self.norm(residual + self.dropout(out))

In [ ]:
"""Composed transformer-to-GAT EEG-to-EMG model variants."""

from __future__ import annotations

import torch
import torch.nn as nn



class KGGTModel(nn.Module):
    """Transformer temporal encoder followed by muscle-graph reasoning."""

    def __init__(
        self,
        transformer_cfg: dict,
        input_dim: int,
        node_dim: int = 64,
        gat_hidden_dim: int = 64,
        gat_heads: int = 4,
        out_channels: int = 5,
        kin_dim: int = 13,
        n_gat_layers: int = 2,
        gat_dropout: float = 0.1,
        use_kinematic_guidance: bool = False,
    ) -> None:
        super().__init__()
        self.use_kinematic_guidance = use_kinematic_guidance
        self.out_channels = out_channels
        self.encoder = build_transformer_from_config(transformer_cfg, input_dim)
        self.node_projection = MuscleNodeProjection(
            input_dim=self.encoder.d_model,
            n_nodes=out_channels,
            node_dim=node_dim,
        )
        if use_kinematic_guidance:
            self.gat = KinematicGuidedMuscleGATEncoder(
                node_dim=node_dim,
                kin_dim=kin_dim,
                hidden_dim=gat_hidden_dim,
                num_heads=gat_heads,
                out_dim=node_dim,
                dropout=gat_dropout,
            )
        else:
            self.gat = MuscleGATEncoder(
                node_dim=node_dim,
                hidden_dim=gat_hidden_dim,
                num_heads=gat_heads,
                out_dim=node_dim,
                n_layers=n_gat_layers,
                dropout=gat_dropout,
            )
        self.decoder = nn.Linear(node_dim, 1)

    def forward(self, eeg: torch.Tensor, kin: torch.Tensor | None = None) -> torch.Tensor:
        temporal = self.encoder(eeg)
        nodes = self.node_projection(temporal)
        if self.use_kinematic_guidance:
            if kin is None:
                raise ValueError("kin is required when use_kinematic_guidance=True")
            refined = self.gat(nodes, kin)
        else:
            refined = self.gat(nodes)
        return self.decoder(refined).squeeze(-1)


def build_kg_gt_from_config(
    cfg: dict,
    input_dim: int,
    kin_dim: int = 13,
) -> KGGTModel:
    """Build a separate KG-GT variant from the project config."""
    model_cfg = cfg["model"]
    gat_cfg = model_cfg.get("gat", {})
    model_type = model_cfg.get("type", "transformer_regressor")
    use_kinematic_guidance = model_type == "kg_gt_kinematic" or gat_cfg.get("use_kinematic_guidance", False)
    return KGGTModel(
        transformer_cfg=model_cfg["transformer"],
        input_dim=input_dim,
        node_dim=gat_cfg.get("node_dim", 64),
        gat_hidden_dim=gat_cfg.get("hidden_dim", 64),
        gat_heads=gat_cfg.get("heads", 4),
        out_channels=model_cfg.get("decoder", {}).get("out_channels", 5),
        kin_dim=kin_dim,
        n_gat_layers=gat_cfg.get("n_layers", 2),
        gat_dropout=gat_cfg.get("dropout", model_cfg["transformer"].get("dropout", 0.1)),
        use_kinematic_guidance=use_kinematic_guidance,
    )

In [ ]:
"""
Batched Soft-DTW loss (Cuturi & Blondel, 2017) for EMG sequence regression.

Soft-DTW replaces the hard min in DTW with a differentiable soft-min, giving a
loss that tolerates small temporal misalignments between predicted and target
EMG envelopes — useful when the decoder gets the shape right but is shifted a
few samples in time, which plain MSE penalises harshly.

Implementation notes (GPU):
  * The DP grid is filled along anti-diagonals. Cells on one anti-diagonal are
    independent, so each of the ~2T steps is a single vectorised op over the
    batch and the diagonal — O(T) Python-level steps instead of O(T^2).
  * Boundaries use a large finite constant (not +inf) so soft-min gradients
    stay finite.
  * Forward only; gradients flow through autograd (no custom backward kernel),
    which is enough for training on the GTX 1660 Ti at W=500.

CombinedEMGLoss mixes MSE and Soft-DTW per the config:
    loss = loss_lambda * MSE + (1 - loss_lambda) * SoftDTW_norm
"""

from __future__ import annotations

import torch
import torch.nn as nn
from torch import Tensor

# Large finite stand-in for +inf on the DP boundary (keeps softmin grad finite).
# NOTE: must be < float16 max (~65504), but the DP always runs in float32
# so 1e9 is fine there; we only need float16-safety if dtype is ever float16.
_BIG = 1.0e9


def _soft_min(a: Tensor, b: Tensor, c: Tensor, gamma: float) -> Tensor:
    """Differentiable soft-min: -gamma * logsumexp(-x/gamma)."""
    stacked = torch.stack((a, b, c), dim=0) / -gamma     # (3, ...)
    return -gamma * torch.logsumexp(stacked, dim=0)


def _squared_euclidean(pred: Tensor, target: Tensor) -> Tensor:
    """Pairwise squared L2 cost matrix over time.

    Args:
        pred:   (B, T, C)
        target: (B, T, C)

    Returns:
        (B, T, T) where D[b, i, j] = ||pred[b, i] - target[b, j]||^2.
    """
    # (B, T, 1, C) - (B, 1, T, C) -> (B, T, T, C) -> sum over C
    diff = pred.unsqueeze(2) - target.unsqueeze(1)
    return (diff * diff).sum(dim=-1)


def soft_dtw(pred: Tensor, target: Tensor, gamma: float = 0.1) -> Tensor:
    """Batched Soft-DTW distance between two equal-length sequences.

    Args:
        pred:   (B, T, C) predicted sequence.
        target: (B, T, C) target sequence.
        gamma:  Soft-min smoothing (smaller -> closer to hard DTW).

    Returns:
        (B,) Soft-DTW value per batch element (unnormalised).
    """
    if pred.shape != target.shape:
        raise ValueError(f"shape mismatch: {tuple(pred.shape)} vs {tuple(target.shape)}")
    if pred.dim() != 3:
        raise ValueError(f"expected (B, T, C), got {tuple(pred.shape)}")

    B, T, _ = pred.shape
    device, dtype = pred.device, pred.dtype

    # The DP accumulates large sentinel values and logsumexp — keep it in
    # float32 even when AMP has downcast inputs to float16 (_BIG=1e9 overflows
    # float16 max of ~65504 and introduces NaNs in the gradient).
    D = _squared_euclidean(pred, target).float()          # (B, T, T) fp32

    # R is 1-indexed with a padded border: shape (B, T+1, T+1).
    R = torch.full((B, T + 1, T + 1), _BIG, device=device, dtype=torch.float32)
    R[:, 0, 0] = 0.0

    # Fill along anti-diagonals d = i + j, for i, j in 1..T.
    for d in range(2, 2 * T + 1):
        i_lo = max(1, d - T)
        i_hi = min(T, d - 1)
        i = torch.arange(i_lo, i_hi + 1, device=device)
        j = d - i

        r0 = R[:, i - 1, j - 1]      # diagonal predecessor
        r1 = R[:, i - 1, j]          # up
        r2 = R[:, i, j - 1]          # left
        cost = D[:, i - 1, j - 1]    # D is 0-indexed
        R[:, i, j] = cost + _soft_min(r0, r1, r2, gamma)

    # Cast back so the scalar loss matches the caller's dtype (e.g. float16
    # under AMP) — autograd handles the upcast transparently.
    return R[:, T, T].to(dtype)


class SoftDTWLoss(nn.Module):
    """Soft-DTW as a reduction-aware loss module.

    Args:
        gamma:      Soft-min smoothing parameter.
        normalize:  Divide by sequence length T so the scale is comparable to
                    a per-step MSE term.
        reduction:  'mean' | 'sum' | 'none' over the batch.
    """

    def __init__(
        self,
        gamma: float = 0.1,
        normalize: bool = True,
        reduction: str = "mean",
    ) -> None:
        super().__init__()
        if reduction not in ("mean", "sum", "none"):
            raise ValueError(f"invalid reduction: {reduction!r}")
        self.gamma = gamma
        self.normalize = normalize
        self.reduction = reduction

    def forward(self, pred: Tensor, target: Tensor) -> Tensor:
        out = soft_dtw(pred, target, self.gamma)          # (B,)
        if self.normalize:
            out = out / pred.size(1)
        if self.reduction == "mean":
            return out.mean()
        if self.reduction == "sum":
            return out.sum()
        return out


class CombinedEMGLoss(nn.Module):
    """Convex mix of MSE and Soft-DTW (method1.tex training objective).

        loss = lambda * MSE + (1 - lambda) * SoftDTW_norm

    With lambda = 1.0 this is plain MSE (Soft-DTW skipped entirely, so no
    O(T^2) grid is built), matching the notebook's current MSE-only behaviour.

    Args:
        loss_lambda:     Weight on the MSE term in [0, 1].
        soft_dtw_gamma:  Soft-min smoothing for the Soft-DTW term.
    """

    def __init__(self, loss_lambda: float = 0.9, soft_dtw_gamma: float = 0.1) -> None:
        super().__init__()
        if not 0.0 <= loss_lambda <= 1.0:
            raise ValueError(f"loss_lambda must be in [0, 1], got {loss_lambda}")
        self.loss_lambda = loss_lambda
        self.mse = nn.MSELoss()
        self.sdtw = SoftDTWLoss(gamma=soft_dtw_gamma, normalize=True, reduction="mean")

    def forward(self, pred: Tensor, target: Tensor) -> Tensor:
        mse = self.mse(pred, target)
        if self.loss_lambda >= 1.0:
            return mse
        return self.loss_lambda * mse + (1.0 - self.loss_lambda) * self.sdtw(pred, target)


def build_loss_from_config(cfg: dict) -> CombinedEMGLoss:
    """Build CombinedEMGLoss from the training config section.

    Args:
        cfg: training dict with loss_lambda and soft_dtw_gamma.
    """
    return CombinedEMGLoss(
        loss_lambda=cfg.get("loss_lambda", 0.9),
        soft_dtw_gamma=cfg.get("soft_dtw_gamma", 0.1),
    )

In [ ]:
"""
Device-aware training loop for the KG-GT EMG regressor.

Runs on CUDA when available (GTX 1660 Ti / sm_75) and falls back to CPU with
the same code path. GPU-specific features:
  * Automatic Mixed Precision (AMP) via torch.amp — roughly halves activation
    memory and uses the Turing FP16 tensor cores, which matters on 6 GB VRAM.
    Disabled automatically on CPU.
  * Gradients scaled with GradScaler to avoid FP16 underflow.

Crash-safe checkpointing:
  * Saves a full checkpoint (weights + optimizer + scheduler + scaler +
    history + epoch) after every epoch to checkpoint_dir/last.pt.
  * Also saves best.pt whenever validation loss improves.
  * On the next call to train_model the loop resumes from last.pt
    automatically (pass resume=True, which is the default when the file exists).

GPU-usage diagnostics:
  * print_gpu_info() logs device name, VRAM capacity, driver/CUDA version.
  * Each epoch log line includes current VRAM allocated/reserved so you can
    confirm the GPU is actually being used.
"""

from __future__ import annotations

import logging
import os
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

try:
    from tqdm.auto import tqdm
    _TQDM_AVAILABLE = True
except ImportError:
    _TQDM_AVAILABLE = False

logger = logging.getLogger(__name__)

ModelInputs = dict[str, Any] | torch.Tensor
PrepareBatch = Callable[[torch.Tensor, torch.Tensor, torch.Tensor], tuple[ModelInputs, torch.Tensor]]


def _forward_model(model: nn.Module, inputs: ModelInputs) -> torch.Tensor:
    """Call the model with either a tensor input or a keyword-input dict."""
    if isinstance(inputs, dict):
        return model(**inputs)
    return model(inputs)


# ---------------------------------------------------------------------------
# GPU diagnostics
# ---------------------------------------------------------------------------

def print_gpu_info(device: torch.device) -> None:
    """Print a clear GPU status block so you can confirm training uses CUDA.

    Call this once before train_model. If the device is CPU this will tell
    you CUDA is unavailable and print the reason (useful for debugging).
    """
    sep = "=" * 60
    print(sep)
    print("  GPU / Device Status")
    print(sep)
    if device.type != "cuda":
        print(f"  WARNING  Running on CPU  (device={device})")
        if not torch.cuda.is_available():
            print("  torch.cuda.is_available() returned False")
            cuda_path = os.environ.get("CUDA_PATH", "<not set>")
            print(f"  CUDA_PATH env var : {cuda_path}")
        else:
            print("  CUDA is available but the caller chose CPU.")
        print(sep)
        return

    idx = device.index if device.index is not None else torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)
    total_mb = props.total_memory / 1024 ** 2
    alloc_mb = torch.cuda.memory_allocated(idx) / 1024 ** 2
    reserv_mb = torch.cuda.memory_reserved(idx) / 1024 ** 2

    print(f"  OK  Device index    : cuda:{idx}")
    print(f"  OK  GPU name        : {props.name}")
    print(f"  OK  CUDA capability : sm_{props.major}{props.minor}")
    print(f"  OK  VRAM (total)    : {total_mb:,.0f} MB")
    print(f"  OK  VRAM allocated  : {alloc_mb:.1f} MB")
    print(f"  OK  VRAM reserved   : {reserv_mb:.1f} MB")
    print(f"  OK  CUDA version    : {torch.version.cuda}")
    print(f"  OK  PyTorch build   : {torch.__version__}")
    has_fp16_hw = props.major >= 7
    amp_note = "YES -- Turing tensor cores active" if has_fp16_hw else "YES (no dedicated FP16 HW)"
    print(f"  OK  AMP (FP16)      : {amp_note}")
    print(sep)


def _gpu_mem_str(device: torch.device) -> str:
    """Return a compact VRAM string e.g. '1234/6144 MB', or empty on CPU."""
    if device.type != "cuda":
        return ""
    idx = device.index if device.index is not None else torch.cuda.current_device()
    alloc = torch.cuda.memory_allocated(idx) / 1024 ** 2
    total = torch.cuda.get_device_properties(idx).total_memory / 1024 ** 2
    return f" | VRAM {alloc:.0f}/{total:.0f} MB"


# ---------------------------------------------------------------------------
# Config / result dataclasses
# ---------------------------------------------------------------------------

@dataclass
class TrainConfig:
    """Resolved training hyperparameters (subset of configs/default.yaml)."""

    lr: float = 1e-3
    lr_patience: int = 50
    lr_factor: float = 0.5
    grad_clip_norm: float = 1.0
    early_stop_patience: int = 30
    max_epochs: int = 500
    use_amp: bool = True
    gradient_accumulation_steps: int = 1
    log_memory_every: int = 50
    # --- checkpoint settings ---
    checkpoint_dir: str = "outputs/checkpoints"
    checkpoint_every: int = 1   # save last.pt every N epochs (1 = every epoch)

    @classmethod
    def from_config(cls, cfg: dict, max_epochs: int | None = None) -> "TrainConfig":
        """Build from the training section of default.yaml.

        Args:
            cfg: training dict.
            max_epochs: override (e.g. small value for a smoke run).
        """
        return cls(
            lr=cfg.get("lr", 1e-3),
            lr_patience=cfg.get("lr_patience", 50),
            lr_factor=cfg.get("lr_factor", 0.5),
            grad_clip_norm=cfg.get("grad_clip_norm", 1.0),
            early_stop_patience=cfg.get("early_stop_patience", 30),
            max_epochs=max_epochs if max_epochs is not None else cfg.get("max_epochs", 500),
            use_amp=cfg.get("use_amp", True),
            gradient_accumulation_steps=max(1, cfg.get("gradient_accumulation_steps", 1)),
            log_memory_every=max(1, cfg.get("log_memory_every", 50)),
            checkpoint_dir=cfg.get("checkpoint_dir", "outputs/checkpoints"),
            checkpoint_every=cfg.get("checkpoint_every", 1),
        )


@dataclass
class TrainResult:
    """Outcome of a training run."""

    best_val: float
    best_state: dict | None
    history: dict[str, list[float]] = field(default_factory=lambda: {"train": [], "val": []})


# ---------------------------------------------------------------------------
# Internal epoch runner
# ---------------------------------------------------------------------------

def _run_epoch(
    model: nn.Module,
    loader: DataLoader,
    prepare_batch: PrepareBatch,
    loss_fn: nn.Module,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None,
    scaler: "torch.amp.GradScaler | None",
    grad_clip: float,
    use_amp: bool,
    gradient_accumulation_steps: int,
    epoch: int,
    phase: str,
) -> float:
    """Run one epoch. Trains when optimizer is given, else evaluates.

    Shows a per-batch tqdm progress bar (if tqdm is installed) so you can
    see real-time throughput rather than waiting for the epoch to complete.
    """
    train = optimizer is not None
    model.train(train)
    amp_device = "cuda" if device.type == "cuda" else "cpu"

    total, n = 0.0, 0
    if train:
        optimizer.zero_grad(set_to_none=True)

    if _TQDM_AVAILABLE:
        bar = tqdm(
            loader,
            desc=f"  Ep {epoch:4d} [{phase:5s}]",
            leave=False,
            unit="batch",
            dynamic_ncols=True,
        )
    else:
        bar = loader

    for batch_idx, (eeg, kin, emg) in enumerate(bar, start=1):
        model_inputs, y = prepare_batch(eeg, kin, emg)
        with torch.set_grad_enabled(train):
            with torch.amp.autocast(device_type=amp_device, enabled=use_amp):
                pred = _forward_model(model, model_inputs)
                loss = loss_fn(pred, y)

            if train:
                loss_for_backward = loss / gradient_accumulation_steps
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss_for_backward).backward()
                else:
                    loss_for_backward.backward()

                should_step = (
                    batch_idx % gradient_accumulation_steps == 0
                    or batch_idx == len(loader)
                )
                if should_step:
                    if scaler is not None and scaler.is_enabled():
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                        optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

        bs = eeg.size(0)
        batch_loss = loss.item()
        total += batch_loss * bs
        n += bs

        if _TQDM_AVAILABLE:
            bar.set_postfix(loss=f"{batch_loss:.4f}")

    return total / max(n, 1)


# ---------------------------------------------------------------------------
# Checkpoint helpers
# ---------------------------------------------------------------------------

def _ckpt_path(ckpt_dir: str, name: str) -> Path:
    p = Path(ckpt_dir)
    p.mkdir(parents=True, exist_ok=True)
    return p / name


def _save_training_checkpoint(
    path: Path,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    best_val: float,
    history: dict,
    bad: int,
    cfg: TrainConfig,
) -> None:
    """Save a full resumable checkpoint (weights + all training state)."""
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "best_val": best_val,
            "history": history,
            "bad_epochs": bad,
            "train_config": cfg.__dict__,
        },
        path,
    )


def _load_training_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    device: torch.device,
) -> tuple[int, float, dict, int]:
    """Load checkpoint and return (start_epoch, best_val, history, bad_epochs)."""
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler is not None and ckpt.get("scaler_state_dict") is not None:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    return (
        ckpt["epoch"],
        ckpt["best_val"],
        ckpt["history"],
        ckpt["bad_epochs"],
    )


# ---------------------------------------------------------------------------
# Public training API
# ---------------------------------------------------------------------------

def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    prepare_batch: PrepareBatch,
    loss_fn: nn.Module,
    device: torch.device,
    cfg: TrainConfig,
    resume: bool = True,
) -> TrainResult:
    """Train with AMP, grad-clip, LR plateau scheduling, and early stopping.

    Crash-safe:
        A ``last.pt`` checkpoint is written to ``cfg.checkpoint_dir`` after
        every ``cfg.checkpoint_every`` epochs. A ``best.pt`` is written
        whenever the validation loss improves. Set ``resume=True`` (default)
        and the loop will pick up from where it left off automatically.

    GPU visibility:
        ``print_gpu_info(device)`` is called at the start so you see a full
        device report. Each epoch log line shows VRAM usage so you can confirm
        the GPU is active throughout training.

    Args:
        model:         Module already moved to ``device``.
        train_loader:  Yields (eeg, kin, emg) batches.
        val_loader:    Validation batches.
        prepare_batch: (eeg, emg) -> (x, y) on device (CCA + EMG z-score).
        loss_fn:       Loss module (e.g. CombinedEMGLoss).
        device:        Target device.
        cfg:           Resolved TrainConfig.
        resume:        If True and ``last.pt`` exists in checkpoint_dir,
                       resume from that checkpoint.

    Returns:
        TrainResult with best val loss, best CPU state_dict, and history.
    """
    use_amp = cfg.use_amp and device.type == "cuda"
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=cfg.lr_factor, patience=cfg.lr_patience
    )
    scaler = torch.amp.GradScaler(enabled=use_amp)

    # Print GPU status so the user can visually confirm CUDA is active
    print_gpu_info(device)

    logger.info(
        "Training on %s | AMP=%s | epochs=%d | lr=%g | ckpt_dir=%s",
        device, use_amp, cfg.max_epochs, cfg.lr, cfg.checkpoint_dir,
    )

    # Resume or fresh start
    last_ckpt = _ckpt_path(cfg.checkpoint_dir, "last.pt")
    start_epoch = 0
    result = TrainResult(best_val=float("inf"), best_state=None)
    bad = 0

    if resume and last_ckpt.exists():
        start_epoch, result.best_val, result.history, bad = _load_training_checkpoint(
            last_ckpt, model, optimizer, scheduler, scaler, device
        )
        logger.info(
            "Resumed from %s  (epoch %d done, best_val=%.4f)",
            last_ckpt, start_epoch, result.best_val,
        )
        print(
            f"\nResumed from checkpoint: {last_ckpt}\n"
            f"  Completed epochs : {start_epoch}\n"
            f"  Best val loss    : {result.best_val:.4f}\n"
            f"  Early-stop bad   : {bad}/{cfg.early_stop_patience}\n"
        )
    else:
        print(f"\nStarting fresh training run  (checkpoint_dir={cfg.checkpoint_dir})\n")

    best_ckpt = _ckpt_path(cfg.checkpoint_dir, "best.pt")

    for ep in range(start_epoch + 1, cfg.max_epochs + 1):
        t0 = time.time()

        tr = _run_epoch(
            model, train_loader, prepare_batch, loss_fn, device,
            optimizer, scaler, cfg.grad_clip_norm, use_amp,
            cfg.gradient_accumulation_steps,
            epoch=ep, phase="train",
        )
        vl = _run_epoch(
            model, val_loader, prepare_batch, loss_fn, device,
            None, None, cfg.grad_clip_norm, use_amp,
            1,
            epoch=ep, phase="val",
        )
        scheduler.step(vl)
        result.history["train"].append(tr)
        result.history["val"].append(vl)

        epoch_time = time.time() - t0
        gpu_mem = _gpu_mem_str(device)
        lr_now = optimizer.param_groups[0]["lr"]

        flag = ""
        if vl < result.best_val:
            result.best_val = vl
            result.best_state = {
                k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            }
            bad = 0
            flag = "  <- BEST"
            # Save best checkpoint immediately on improvement
            _save_training_checkpoint(
                best_ckpt, ep, model, optimizer, scheduler, scaler,
                result.best_val, result.history, bad, cfg,
            )
        else:
            bad += 1

        summary = (
            f"Ep {ep:4d}/{cfg.max_epochs} | "
            f"train {tr:.4f} | val {vl:.4f} | "
            f"lr {lr_now:.2e} | "
            f"{epoch_time:5.1f}s"
            f"{gpu_mem}"
            f"{flag}"
        )
        print(summary)
        logger.info(summary)

        # Periodic crash-safe checkpoint
        if ep % cfg.checkpoint_every == 0:
            _save_training_checkpoint(
                last_ckpt, ep, model, optimizer, scheduler, scaler,
                result.best_val, result.history, bad, cfg,
            )

        # Early stopping
        if bad >= cfg.early_stop_patience:
            msg = f"Early stop at epoch {ep} (no val improvement for {bad} epochs)."
            print(msg)
            logger.info(msg)
            # Always write last.pt on stop
            _save_training_checkpoint(
                last_ckpt, ep, model, optimizer, scheduler, scaler,
                result.best_val, result.history, bad, cfg,
            )
            break

    if result.best_state is not None:
        model.load_state_dict(result.best_state)

    print(f"\nTraining done.  Best val loss = {result.best_val:.4f}")
    print(f"Checkpoints saved to: {Path(cfg.checkpoint_dir).resolve()}")
    return result


# ---------------------------------------------------------------------------
# Standalone inference checkpoint helpers
# ---------------------------------------------------------------------------

def save_checkpoint(path: str, model: nn.Module, cfg: TrainConfig, best_val: float) -> None:
    """Save model weights + minimal metadata for inference.

    This is a lightweight export checkpoint (weights only). For full
    resumable checkpoints use the last.pt / best.pt files written
    automatically by train_model.
    """
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "train_config": cfg.__dict__,
            "best_val": best_val,
        },
        path,
    )
    logger.info("checkpoint saved -> %s", path)
    print(f"Saved inference checkpoint -> {path}")


def load_checkpoint(path: str, model: nn.Module, device: torch.device) -> float:
    """Load weights from a save_checkpoint file.  Returns best_val."""
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    best_val = ckpt.get("best_val", float("inf"))
    logger.info("loaded checkpoint <- %s  (best_val=%.4f)", path, best_val)
    print(f"Loaded checkpoint <- {path}  (best_val={best_val:.4f})")
    return best_val

In [ ]:
"""
Evaluation metrics for the KG-GT EMG regressor.

Collects predictions over a DataLoader on-device, then reports per-channel
RMSE, MAE, and Pearson r on the (z-scored) EMG targets. Inference runs under
autocast on CUDA for speed; metrics are computed in float64 on CPU for
numerical stability.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Any, Callable

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

logger = logging.getLogger(__name__)

ModelInputs = dict[str, Any] | torch.Tensor
PrepareBatch = Callable[[torch.Tensor, torch.Tensor, torch.Tensor], tuple[ModelInputs, torch.Tensor]]


def _forward_model(model: nn.Module, inputs: ModelInputs) -> torch.Tensor:
    """Call the model with either a tensor input or a keyword-input dict."""
    if isinstance(inputs, dict):
        return model(**inputs)
    return model(inputs)


@dataclass
class EvalMetrics:
    """Per-channel and mean regression metrics."""

    rmse: np.ndarray   # (C,)
    mae: np.ndarray    # (C,)
    pearson: np.ndarray  # (C,)
    channel_names: list[str]

    def as_table(self) -> str:
        """Format metrics as an aligned text table."""
        lines = [f'{"channel":20s} {"RMSE":>8s} {"MAE":>8s} {"Pearson":>8s}']
        for c, name in enumerate(self.channel_names):
            lines.append(
                f"{name:20s} {self.rmse[c]:8.3f} {self.mae[c]:8.3f} {self.pearson[c]:8.3f}"
            )
        lines.append(
            f'{"MEAN":20s} {self.rmse.mean():8.3f} '
            f"{self.mae.mean():8.3f} {self.pearson.mean():8.3f}"
        )
        return "\n".join(lines)


@torch.no_grad()
def collect_predictions(
    model: nn.Module,
    loader: DataLoader,
    prepare_batch: PrepareBatch,
    device: torch.device,
    n_channels: int = 5,
    use_amp: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Run the model over a loader and return stacked (pred, target).

    Returns:
        (P, Y) each (N_total, n_channels) float32 numpy arrays.
    """
    model.eval()
    amp_device = "cuda" if device.type == "cuda" else "cpu"
    amp = use_amp and device.type == "cuda"

    preds, targets = [], []
    for eeg, kin, emg in loader:
        model_inputs, y = prepare_batch(eeg, kin, emg)
        with torch.amp.autocast(device_type=amp_device, enabled=amp):
            pred = _forward_model(model, model_inputs)
        preds.append(pred.float().cpu().numpy().reshape(-1, n_channels))
        targets.append(y.float().cpu().numpy().reshape(-1, n_channels))

    return np.concatenate(preds), np.concatenate(targets)


def compute_metrics(
    pred: np.ndarray,
    target: np.ndarray,
    channel_names: list[str] | None = None,
) -> EvalMetrics:
    """Per-channel RMSE, MAE, Pearson r between pred and target.

    Args:
        pred:   (N, C)
        target: (N, C)
        channel_names: optional labels, defaults to ch0..ch{C-1}.
    """
    pred = pred.astype(np.float64)
    target = target.astype(np.float64)
    n_channels = pred.shape[1]

    if channel_names is None:
        channel_names = [f"ch{c}" for c in range(n_channels)]

    rmse = np.sqrt(((pred - target) ** 2).mean(0))
    mae = np.abs(pred - target).mean(0)
    pearson = np.array(
        [np.corrcoef(pred[:, c], target[:, c])[0, 1] for c in range(n_channels)]
    )
    return EvalMetrics(rmse=rmse, mae=mae, pearson=pearson, channel_names=channel_names)


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    prepare_batch: PrepareBatch,
    device: torch.device,
    channel_names: list[str] | None = None,
    n_channels: int = 5,
    use_amp: bool = True,
) -> EvalMetrics:
    """End-to-end evaluation: collect predictions then compute metrics."""
    pred, target = collect_predictions(
        model, loader, prepare_batch, device, n_channels=n_channels, use_amp=use_amp
    )
    metrics = compute_metrics(pred, target, channel_names)
    logger.info("evaluation complete:\n%s", metrics.as_table())
    return metrics

In [ ]:
# @title PaperStylePlotHelpers
PAPER_STYLE = {
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "grid.linestyle": "--",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 120,
}
plt.rcParams.update(PAPER_STYLE)
sns.set_theme(style="whitegrid", context="paper")


def _make_time_axis(data, fs):
    return np.arange(len(data)) / float(fs)


def plot_multichannel_trace(data, fs=500, channel_names=None, title="Signal", max_channels=8, figsize=(12, 6)):
    data = np.asarray(data)
    if data.ndim == 1:
        data = data[:, None]
    n_channels = min(data.shape[1], max_channels)
    time = _make_time_axis(data, fs)
    fig, axes = plt.subplots(n_channels, 1, figsize=figsize, sharex=True)
    if n_channels == 1:
        axes = [axes]
    for idx, ax in enumerate(axes):
        ax.plot(time, data[:, idx], linewidth=1.1, color=f"C{idx % 10}")
        ax.set_ylabel(channel_names[idx] if channel_names else f"ch{idx}")
        ax.grid(True, alpha=0.2)
    axes[0].set_title(title)
    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


def plot_signal_heatmap(data, title="Signal Heatmap", channel_names=None, figsize=(12, 4), cmap="mako"):
    data = np.asarray(data)
    plt.figure(figsize=figsize)
    sns.heatmap(data.T, cmap=cmap, cbar=True)
    plt.title(title)
    plt.xlabel("Samples")
    plt.ylabel("Channels")
    if channel_names and len(channel_names) == data.shape[1]:
        plt.yticks(np.arange(len(channel_names)) + 0.5, channel_names, rotation=0)
    plt.tight_layout()
    plt.show()


def plot_preprocessing_comparison(raw_data, processed_data, fs=500, channel_idx=0, title_prefix="EEG", figsize=(12, 6)):
    raw_data = np.asarray(raw_data)
    processed_data = np.asarray(processed_data)
    time_raw = _make_time_axis(raw_data, fs)
    time_proc = _make_time_axis(processed_data, fs)
    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=False)
    axes[0].plot(time_raw, raw_data[:, channel_idx], color="tab:gray", linewidth=1.0)
    axes[0].set_title(f"{title_prefix} Raw - channel {channel_idx}")
    axes[1].plot(time_proc, processed_data[:, channel_idx], color="tab:blue", linewidth=1.0)
    axes[1].set_title(f"{title_prefix} Processed - channel {channel_idx}")
    axes[1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


def plot_kinematic_features(kin, fs=500, feature_names=None, max_features=6, figsize=(12, 8)):
    kin = np.asarray(kin)
    n_features = min(kin.shape[1], max_features)
    time = _make_time_axis(kin, fs)
    fig, axes = plt.subplots(n_features, 1, figsize=figsize, sharex=True)
    if n_features == 1:
        axes = [axes]
    for idx, ax in enumerate(axes):
        ax.plot(time, kin[:, idx], linewidth=1.0, color=f"C{idx % 10}")
        ax.set_ylabel(feature_names[idx] if feature_names else f"k{idx}")
    axes[0].set_title("Kinematic State Features")
    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


def plot_cca_correlations(correlations, title="CCA Canonical Correlations", figsize=(8, 4)):
    correlations = np.asarray(correlations)
    plt.figure(figsize=figsize)
    xs = np.arange(1, len(correlations) + 1)
    bars = plt.bar(xs, correlations, color="#4C78A8", edgecolor="black", linewidth=0.6)
    for bar, value in zip(bars, correlations):
        plt.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.2f}", ha="center", va="bottom", fontsize=9)
    plt.title(title)
    plt.xlabel("Component")
    plt.ylabel("Correlation")
    plt.ylim(0, min(1.0, correlations.max() + 0.15))
    plt.tight_layout()
    plt.show()


def plot_cca_scatter(x_scores, y_scores, component=0, figsize=(6, 6)):
    x = np.asarray(x_scores)[:, component]
    y = np.asarray(y_scores)[:, component]
    r = np.corrcoef(x, y)[0, 1]
    plt.figure(figsize=figsize)
    sns.scatterplot(x=x, y=y, s=18, alpha=0.5, edgecolor=None)
    sns.regplot(x=x, y=y, scatter=False, color="crimson")
    plt.title(f"CCA Component {component + 1} (r={r:.3f})")
    plt.xlabel("EEG canonical score")
    plt.ylabel("Kinematic canonical score")
    plt.tight_layout()
    plt.show()


def plot_cca_timeseries(x_scores, y_scores, components=3, fs=500, figsize=(12, 8)):
    components = min(components, x_scores.shape[1], y_scores.shape[1])
    time = _make_time_axis(x_scores, fs)
    fig, axes = plt.subplots(components, 1, figsize=figsize, sharex=True)
    if components == 1:
        axes = [axes]
    for idx, ax in enumerate(axes):
        x = x_scores[:, idx]
        y = y_scores[:, idx]
        x = (x - x.mean()) / (x.std() + 1e-8)
        y = (y - y.mean()) / (y.std() + 1e-8)
        ax.plot(time, x, label="EEG score", linewidth=1.1)
        ax.plot(time, y, label="Kinematic score", linewidth=1.1, linestyle="--")
        ax.set_title(f"CCA component {idx + 1}")
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


def plot_training_history(history, figsize=(10, 4)):
    plt.figure(figsize=figsize)
    plt.plot(history.get("train", []), label="Train loss", linewidth=1.8)
    plt.plot(history.get("val", []), label="Val loss", linewidth=1.8)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training History")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_prediction_overlay(pred, target, fs=500, channel_names=None, channels=None, figsize=(12, 8)):
    pred = np.asarray(pred)
    target = np.asarray(target)
    if channels is None:
        channels = list(range(min(pred.shape[1], 5)))
    time = _make_time_axis(pred, fs)
    fig, axes = plt.subplots(len(channels), 1, figsize=figsize, sharex=True)
    if len(channels) == 1:
        axes = [axes]
    for ax, ch in zip(axes, channels):
        ax.plot(time, target[:, ch], label="Target", linewidth=1.2, color="black")
        ax.plot(time, pred[:, ch], label="Prediction", linewidth=1.0, color="tab:red", alpha=0.85)
        ax.set_ylabel(channel_names[ch] if channel_names else f"EMG {ch}")
        ax.legend(loc="upper right")
    axes[0].set_title("Prediction vs Target")
    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


def plot_residual_diagnostics(pred, target, channel_idx=0, figsize=(12, 4)):
    residual = np.asarray(pred)[:, channel_idx] - np.asarray(target)[:, channel_idx]
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].plot(residual, linewidth=0.9)
    axes[0].set_title(f"Residual trace - channel {channel_idx}")
    sns.histplot(residual, kde=True, ax=axes[1], color="tab:purple")
    axes[1].set_title(f"Residual distribution - channel {channel_idx}")
    plt.tight_layout()
    plt.show()


def plot_metric_bars(metric_values, channel_names, title, ylabel, figsize=(9, 4)):
    metric_values = np.asarray(metric_values)
    plt.figure(figsize=figsize)
    sns.barplot(x=channel_names, y=metric_values, palette="crest")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xlabel("Channel")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


def plot_gat_attention_heatmap(attention_matrix, node_names=None, title="GAT Attention", figsize=(5, 4), cmap="rocket"):
    attention_matrix = np.asarray(attention_matrix)
    plt.figure(figsize=figsize)
    sns.heatmap(attention_matrix, annot=True, fmt=".2f", cmap=cmap, cbar=True,
                xticklabels=node_names, yticklabels=node_names)
    plt.title(title)
    plt.xlabel("Source node")
    plt.ylabel("Target node")
    plt.tight_layout()
    plt.show()

In [ ]:
# @title PipelineHelpers
EMG_CHANNEL_NAMES = [
    "Ant. Deltoid",
    "Ext. Carpi Rad.",
    "Flex. Digitorum",
    "Ext. Dig. Comm.",
    "1st Dors. Inteross.",
]


def make_preprocess_fn(cfg):
    eeg_cfg = cfg["preprocessing"]["eeg"]
    emg_cfg = cfg["preprocessing"]["emg"]

    def preprocess_fn(series):
        series = dict(series)
        series["eeg"] = preprocess_eeg_from_config(
            series["eeg"],
            float(series["fs_eeg"]),
            eeg_cfg,
            channel_names=series.get("eeg_names"),
        )
        series["emg"] = preprocess_emg_from_config(
            series["emg"],
            float(series["fs_emg"]),
            emg_cfg,
        )
        return series

    return preprocess_fn


def build_dataset_split(cfg, participants=None, split="train", root_dir=ROOT):
    data_cfg = cfg["data"]
    if participants is None:
        participants = data_cfg["participants"]
    return WAYEEGDataset(
        data_dir=root_dir / data_cfg["raw_dir"],
        participants=participants,
        split=split,
        window_size=data_cfg["window_size"],
        stride=data_cfg["stride"],
        preprocess_fn=make_preprocess_fn(cfg),
        cache_dir=root_dir / data_cfg["cache_dir"],
    )


def unique_series_arrays(ds):
    seen, eegs, kins, emgs = set(), [], [], []
    for eeg_all, kin_all, emg_all, _ in ds._windows:
        key = id(eeg_all)
        if key in seen:
            continue
        seen.add(key)
        eegs.append(eeg_all)
        kins.append(kin_all)
        emgs.append(emg_all)
    return eegs, kins, emgs


def fit_cca_and_emg_stats(train_ds, cca_cfg, device):
    tr_eeg, tr_kin, tr_emg = unique_series_arrays(train_ds)
    cca = make_cca_from_config(cca_cfg)
    cca.fit(tr_eeg, tr_kin)
    cca_gpu = cca.torch_projector(device)

    emg_concat = np.concatenate(tr_emg, axis=0)
    emg_mean = torch.tensor(emg_concat.mean(0), dtype=torch.float32, device=device)
    emg_std = torch.tensor(emg_concat.std(0) + 1e-8, dtype=torch.float32, device=device)
    return cca, cca_gpu, emg_mean, emg_std


def prepare_batch_factory(cca_projector, emg_mean, emg_std, device, n_cca):
    def prepare_batch(eeg, kin, emg):
        batch_size, window_size, n_features = eeg.shape
        flat = eeg.reshape(-1, n_features).to(device, non_blocking=True)
        proj = cca_projector.transform(flat).reshape(batch_size, window_size, n_cca)
        kin_device = kin.to(device, non_blocking=True)
        emg_norm = (emg.to(device, non_blocking=True) - emg_mean) / emg_std
        return {"eeg": proj, "kin": kin_device}, emg_norm

    return prepare_batch


def build_model_from_config(cfg, input_dim, kin_dim=13):
    model_type = cfg["model"].get("type", "transformer_regressor")
    if model_type == "kg_gt":
        cfg = copy.deepcopy(cfg)
        cfg["model"]["gat"]["use_kinematic_guidance"] = False
        return build_kg_gt_from_config(cfg, input_dim=input_dim, kin_dim=kin_dim)
    if model_type == "kg_gt_kinematic":
        cfg = copy.deepcopy(cfg)
        cfg["model"]["gat"]["use_kinematic_guidance"] = True
        return build_kg_gt_from_config(cfg, input_dim=input_dim, kin_dim=kin_dim)
    raise ValueError("This notebook is focused on Method 1 GAT variants. Set model.type to 'kg_gt' or 'kg_gt_kinematic'.")


def inverse_emg_zscore(arr, emg_mean, emg_std):
    if torch.is_tensor(arr):
        arr = arr.detach().cpu().numpy()
    mean = emg_mean.detach().cpu().numpy() if torch.is_tensor(emg_mean) else np.asarray(emg_mean)
    std = emg_std.detach().cpu().numpy() if torch.is_tensor(emg_std) else np.asarray(emg_std)
    return arr * std + mean

## Recommended Notebook Flow

1. Inspect one raw series
2. Visualize preprocessing outputs
3. Fit CCA on the train split
4. Build the GAT-enabled Method 1 model
5. Run a small smoke train or a full train
6. Visualize prediction quality and channel-wise metrics

In [ ]:
# @title ConfigureNotebookExperiment
notebook_cfg = copy.deepcopy(cfg)

# Keep the current practical training choice from your config.
notebook_cfg["training"]["loss_lambda"] = cfg["training"]["loss_lambda"]

# Method 1 notebook should use the separate GAT pipeline.
notebook_cfg["model"]["type"] = "kg_gt_kinematic"

# These can be edited in-notebook for quick experiments.
participants = notebook_cfg["data"]["participants"]
batch_size = notebook_cfg["training"]["batch_size"]
max_epochs = min(10, notebook_cfg["training"]["max_epochs"])
smoke_run = True

device = get_device(prefer_cuda=True)
set_seed(notebook_cfg["training"].get("seed", 42), deterministic=False)
print("Device:", device)
print("Notebook model type:", notebook_cfg["model"]["type"])
print("Loss lambda:", notebook_cfg["training"]["loss_lambda"])
print("Smoke run:", smoke_run)

In [ ]:
# @title InspectRawParticipantSeries
raw_dir = ROOT / notebook_cfg["data"]["raw_dir"]
participant = participants[0]
series_list = load_participant(raw_dir, participant=participant, file_type="hs", series=[1], include_stability=False)
series = series_list[0]

print("Participant:", series["participant"])
print("Series:", series["series"])
print("EEG shape:", series["eeg"].shape)
print("EMG shape:", series["emg"].shape)
print("Kin shape:", series["kin"].shape)

plot_multichannel_trace(series["eeg"][:2000], fs=series["fs_eeg"], channel_names=series["eeg_names"], title="Raw EEG preview", max_channels=6)
plot_multichannel_trace(series["emg"][:4000], fs=series["fs_emg"], channel_names=series["emg_names"], title="Raw EMG preview", max_channels=5)

In [ ]:
# @title VisualizePreprocessingPipeline
eeg_processed = preprocess_eeg_from_config(
    series["eeg"],
    fs=series["fs_eeg"],
    cfg=notebook_cfg["preprocessing"]["eeg"],
    channel_names=series["eeg_names"],
)
emg_processed = preprocess_emg_from_config(
    series["emg"],
    fs=series["fs_emg"],
    cfg=notebook_cfg["preprocessing"]["emg"],
)
kin_features = preprocess_kinematics_from_config(
    series["kin"],
    fs=series["fs_kin"],
    cfg=notebook_cfg["preprocessing"]["kinematics"],
)

plot_preprocessing_comparison(series["eeg"][:2000], eeg_processed[:2000], fs=series["fs_eeg"], channel_idx=0, title_prefix="EEG")
plot_preprocessing_comparison(series["emg"][:2000], emg_processed[:2000], fs=series["fs_eeg"], channel_idx=0, title_prefix="EMG envelope")
plot_signal_heatmap(eeg_processed[:1200], title="Processed EEG heatmap", cmap="viridis")
plot_signal_heatmap(emg_processed[:1200], title="Processed EMG heatmap", channel_names=series["emg_names"], cmap="magma")
plot_kinematic_features(kin_features[:1500], fs=series["fs_kin"], max_features=6)

In [ ]:
# @title BuildDatasets
train_ds = build_dataset_split(notebook_cfg, participants=participants, split="train", root_dir=ROOT)
val_ds = build_dataset_split(notebook_cfg, participants=participants, split="val", root_dir=ROOT)
test_ds = build_dataset_split(notebook_cfg, participants=participants, split="test", root_dir=ROOT)

print(train_ds)
print(val_ds)
print(test_ds)

In [ ]:
# @title FitCCAAndVisualize
cca_cpu, cca_gpu, emg_mean, emg_std = fit_cca_and_emg_stats(
    train_ds,
    notebook_cfg["preprocessing"]["cca"],
    device,
)

train_eeg_unique, train_kin_unique, _ = unique_series_arrays(train_ds)
sample_eeg = train_eeg_unique[0]
sample_kin = train_kin_unique[0]

eeg_scores, kin_scores = cca_cpu._cca.transform(
    sample_eeg.astype(np.float64),
    sample_kin.astype(np.float64),
)
canonical_corrs = [
    np.corrcoef(eeg_scores[:, i], kin_scores[:, i])[0, 1]
    for i in range(min(eeg_scores.shape[1], kin_scores.shape[1]))
]

plot_cca_correlations(canonical_corrs)
plot_cca_scatter(eeg_scores, kin_scores, component=0)
plot_cca_timeseries(eeg_scores, kin_scores, components=min(3, len(canonical_corrs)))

In [ ]:
# @title BuildMethod1Model
model = build_model_from_config(
    notebook_cfg,
    input_dim=notebook_cfg["preprocessing"]["cca"]["n_components"],
    kin_dim=notebook_cfg["data"].get("n_kin_features", 13),
).to(device)

loss_fn = build_loss_from_config(notebook_cfg["training"]).to(device)
prepare_batch = prepare_batch_factory(
    cca_projector=cca_gpu,
    emg_mean=emg_mean,
    emg_std=emg_std,
    device=device,
    n_cca=notebook_cfg["preprocessing"]["cca"]["n_components"],
)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print("Trainable parameters:", f"{param_count:,}")
print("Loss:", loss_fn)

In [ ]:
# @title ForwardPassSmokeTest
sample_loader = DataLoader(train_ds, batch_size=2, shuffle=False)
sample_eeg, sample_kin, sample_emg = next(iter(sample_loader))
model_inputs, y = prepare_batch(sample_eeg, sample_kin, sample_emg)

with torch.no_grad():
    pred = model(**model_inputs)

print("Projected EEG shape:", model_inputs["eeg"].shape)
print("Kinematics shape:", model_inputs["kin"].shape)
print("Prediction shape:", pred.shape)
print("Target shape:", y.shape)

In [ ]:
# @title TrainMethod1Model
train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=device.type == "cuda",
    num_workers=notebook_cfg["runtime"].get("num_workers", 0),
    persistent_workers=bool(notebook_cfg["runtime"].get("persistent_workers", False) and notebook_cfg["runtime"].get("num_workers", 0) > 0),
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=device.type == "cuda",
    num_workers=notebook_cfg["runtime"].get("num_workers", 0),
    persistent_workers=bool(notebook_cfg["runtime"].get("persistent_workers", False) and notebook_cfg["runtime"].get("num_workers", 0) > 0),
)

train_cfg = TrainConfig.from_config(notebook_cfg["training"], max_epochs=max_epochs if smoke_run else notebook_cfg["training"]["max_epochs"])
train_cfg.checkpoint_dir = str(ROOT / "outputs" / "checkpoints" / "notebook_method1")

result = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    prepare_batch=prepare_batch,
    loss_fn=loss_fn,
    device=device,
    cfg=train_cfg,
    resume=False,
)

In [ ]:
# @title EvaluateMethod1Model
test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=device.type == "cuda",
    num_workers=notebook_cfg["runtime"].get("num_workers", 0),
    persistent_workers=bool(notebook_cfg["runtime"].get("persistent_workers", False) and notebook_cfg["runtime"].get("num_workers", 0) > 0),
)

metrics = evaluate(
    model=model,
    loader=test_loader,
    prepare_batch=prepare_batch,
    device=device,
    channel_names=EMG_CHANNEL_NAMES,
    n_channels=5,
    use_amp=train_cfg.use_amp,
)

print(metrics.as_table())
plot_training_history(result.history)
plot_metric_bars(metrics.rmse, metrics.channel_names, title="Per-channel RMSE", ylabel="RMSE")
plot_metric_bars(metrics.mae, metrics.channel_names, title="Per-channel MAE", ylabel="MAE")
plot_metric_bars(metrics.pearson, metrics.channel_names, title="Per-channel Pearson Correlation", ylabel="r")

In [ ]:
# @title PredictionVisualization
pred_z, target_z = collect_predictions(
    model=model,
    loader=test_loader,
    prepare_batch=prepare_batch,
    device=device,
    n_channels=5,
    use_amp=train_cfg.use_amp,
)

pred = inverse_emg_zscore(pred_z, emg_mean, emg_std)
target = inverse_emg_zscore(target_z, emg_mean, emg_std)

window = min(len(pred), 1000)
plot_prediction_overlay(pred[:window], target[:window], channel_names=EMG_CHANNEL_NAMES)
plot_residual_diagnostics(pred[:window], target[:window], channel_idx=0)

In [ ]:
# @title OptionalGATAttentionVisualization
with torch.no_grad():
    temporal = model.encoder(model_inputs["eeg"])
    nodes = model.node_projection(temporal)
    if getattr(model, "use_kinematic_guidance", False):
        flat_nodes = nodes.reshape(-1, nodes.shape[2], nodes.shape[3])
        flat_kin = model_inputs["kin"].reshape(-1, model_inputs["kin"].shape[-1])
        node_proj = model.gat.node_proj(flat_nodes).view(flat_nodes.shape[0], flat_nodes.shape[1], model.gat.num_heads, model.gat.hidden_dim).permute(0, 2, 1, 3)
        kin_proj = model.gat.kin_proj(flat_kin).view(flat_nodes.shape[0], model.gat.num_heads, model.gat.hidden_dim).unsqueeze(2)
        guided = node_proj + kin_proj
        scores = torch.matmul(guided, guided.transpose(-2, -1)) * model.gat.scale
        attn = torch.softmax(scores, dim=-1)
        attn_mean = attn.mean(dim=(0, 1)).cpu().numpy()
        plot_gat_attention_heatmap(attn_mean, node_names=EMG_CHANNEL_NAMES, title="Mean Kinematic-Guided GAT Attention")
    else:
        print("Attention visualization cell currently targets the kinematic-guided GAT variant.")

## Notes

- This notebook is intentionally **self-contained** and does not import project modules from `src`.
- It reflects the current repo behavior where **`loss_lambda = 1.0`** keeps training MSE-only for practical speed.
- If you later move back to hybrid loss, the inlined `SoftDTWLoss` and `CombinedEMGLoss` cells are already present.